# BioMed-VITAL (13B, 60K) QLoRA 근사 재현 (Expert-CFG Table 1의 BioMed-VITAL 행)

> **동일 재현이 아닙니다.** 공개된 것은 instruction-tuned 가중치 하나(HF `mao1207/BioMed-VITAL-models` @ `c93f260`)와
> 저자 추론 코드(`eval/model_vqa_med.py`)뿐입니다. VQA-RAD·SLAKE로 fine-tuning한 가중치와 fine-tuning 코드는 공개되지 않았습니다.
> 이 노트북은 그 가중치에서 **QLoRA로 downstream fine-tuning**을 하는 근사 실험입니다.

| Dataset | split (train / test) | 논문 Open / Closed / Overall |
|---|---|---|
| VQA-RAD | **원본 1,797 / 451** (MEVF `frame` 질문 제외, 논문 Table 3과 같음) | 64.88 / 84.55 / 76.74 |
| SLAKE 영어 | 4,919 / 1,061 | 87.82 / 86.54 / 87.32 |

**실행 순서 (자동)**

1. (선택, 기본 끔) fine-tuning 전 VQA-RAD test 평가 (zero-shot 기준점)
2. VQA-RAD: LoRA lr **2e-5 / 1e-4 / 2e-4** 각각 **3 epoch cosine 스케줄로 시작해 1 epoch**만 학습 → validation 평가
3. validation이 가장 좋은 lr을 골라 **저장된 상태 그대로 epoch 3까지 이어서** 학습 (처음부터 3 epoch 돌린 것과 같음) → test 평가
4. SLAKE: 고른 lr로 3 epoch → test 평가

**무엇이 같고 무엇이 다른가**

| 항목 | 이 노트북 | 근거 |
|---|---|---|
| 시작 가중치 | BioMed-VITAL instruction-tuned 13B (LLaMA-13B + Linear projector + CLIP ViT-L/14-336) | HF revision 고정 |
| 학습 방식 | **QLoRA**: 4bit NF4 + double quant, bf16 계산, LoRA r 128 / alpha 256 / dropout 0.05 (LLM의 모든 linear), projector 학습(lr 2e-5), CLIP 고정 | 논문은 전체 fine-tuning → LLaVA v1.5 `finetune_task_lora.sh` |
| epoch / batch / 스케줄 | **3 epoch**, 전역 batch 64, cosine, warmup 0.03, wd 0, grad clip 1.0 | 논문 15 epoch → LLaVA-Med v1.0.0 fine-tuning 명령 값 |
| lr | LoRA lr은 3개 중 validation으로 선택, projector 2e-5 | 교수님 지시 / 논문 2e-5는 전체 fine-tuning용 |
| 학습 텍스트·라벨 마스킹 | LLaVA-Med v1.0.0 `train.py` 그대로 (`### Human: 질문\n<image>` / `### Assistant: 답`) | 저자 fine-tuning 데이터 비공개 |
| 이미지 토큰 | `<im_start>` + `<im_patch>`×576 + `<im_end>`, CLIP 뒤에서 두 번째 층, CLS 제외 | LLaVA-Med v1 `LlavaLlamaModel` |
| 추론 프롬프트·디코딩 | 저자 `model_vqa_med.py` (conv `default`, 샘플링 T=0.7, 최대 1024 토큰, `###`에서 중단) | 저자 코드 |
| validation | VQA-RAD train 이미지의 10% (seed 0, 이미지 단위) → train 1,626 / val 171 | 논문에 없음, test는 마지막에만 사용 |
| 채점 | **논문 방식**(LLaVA-Med `run_eval.py`: closed yes/no hit, open token recall) + **통일 방식**(정규화 EM, token F1) | 다른 모델과 비교용 |

Open/Closed는 모든 채점에서 정답 `answer_type` 기준입니다. Overall은 질문 전체 평균입니다(논문 표와 같은 가중).

**필요 자원 (추정)**: A100(40GB 이상, bf16). 모델 다운로드 약 52GB + 디스크 여유 90GB 이상.
전체 약 4~5.5시간(약 50~65 컴퓨팅 단위). 실제 속도는 학습 로그의 `s/step`과 `eta`로 확인하세요.

학습은 **백그라운드 프로세스**로 돕니다. 셀을 중단하거나 출력이 끊겨도 계속됩니다.
optimizer step 10번마다와 epoch마다 상태를 저장하므로, 프로세스가 죽으면 **같은 런타임에서** 5-1 셀을 다시 실행해 이어서 학습합니다.

**저장 (체크포인트·모델)**: VS Code에서 Colab 커널을 쓰면 파일은 PC가 아니라 Colab 런타임 디스크(`/content`)에 생기고,
런타임이 끝나면 사라집니다. 그래서 epoch마다 아래 파일을 **본인 HF 계정의 비공개 model repo**에 자동으로 올리고,
PC에서는 8번 셀의 명령으로 받습니다.

| 파일 | 내용 | 크기 |
|---|---|---|
| `runs/<run>/epoch<k>/` | fine-tuned 모델: LoRA adapter(`adapter_model.safetensors`, peft 형식) + `mm_projector.safetensors` + `meta.json` | 약 2GB |
| `runs/<run>/state.pt` | 이어서 학습용 체크포인트(LoRA·projector·optimizer·scheduler·난수 상태) | 약 6GB |
| `results/*.json`, `summary.json`, `train.log` | 결과(질문별 예측 포함)와 로그 | 작음 |

새 런타임에서 같은 `HUB_REPO`로 5-1을 실행하면 Hub에서 체크포인트와 결과를 먼저 받아 **끝난 단계는 건너뛰고 이어서** 학습합니다.


## 1. 환경 설치

버전을 고정합니다. 설치 후 런타임을 다시 시작하라는 메시지가 나오면 다시 시작한 뒤 2번 셀부터 실행하세요.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.46.3', 'peft==0.13.2',
                       'accelerate==1.1.1', 'bitsandbytes', 'sentencepiece'])
import shutil, torch, transformers, peft, bitsandbytes
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경 > A100 GPU'
assert torch.cuda.is_bf16_supported(), 'bf16이 되는 GPU(A100/L4/H100)가 필요합니다'
free_gb = shutil.disk_usage('/content').free / 1e9
print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__)
print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB', f'disk free {free_gb:.0f} GB')
assert free_gb > 90, '디스크 여유가 90GB 이상 필요합니다'


## 2. 코드

저장소의 `qcr_port.py`(데이터 고정·검증), `biomedclip_meter.py`(정규화 함수), `biomed_vital_qlora.py`(모델·학습·평가)가 그대로 들어 있습니다.


In [ ]:
from pathlib import Path
import sys, importlib
CODE_ROOT = Path('/content/biomed_vital_code')
CODE_ROOT.mkdir(exist_ok=True)
(CODE_ROOT / 'qcr_port.py').write_text('"""Run the official PubMedCLIP QCR code on a current Colab GPU runtime.\n\nThe official code (sarahESL/PubMedCLIP, QCR_PubMedCLIP, pinned commit) is cloned and\nonly environment-compatibility edits are applied (see PATCHES). Model, loss, optimizer,\ndata preprocessing scripts and configs are the authors\' own. Every external asset is\npinned to an exact revision (HF revision or git commit) and verified by SHA-256.\n"""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nQCR_REPO = \'https://github.com/sarahESL/PubMedCLIP.git\'\nQCR_COMMIT = \'bdecf3c5a21b68200a02dc05455b3882a23206a2\'\n\n# repo, repo_type, revision, filename, sha256\nASSETS = {\n    \'mevf_rad\': (\'aiozai/MEVF\', \'dataset\', \'7b3bd1b1ef8965eb35ff61a1f6fada25377765ea\', \'data_RAD.zip\',\n                 \'b75d664c3dac7096a695e3a8a2b309769804f5add7e72b15e037ea9e5b839a9f\'),\n    \'slake_imgs\': (\'BoKelvin/SLAKE\', \'dataset\', \'a9083ce6c34ac3ffb17671a605962924d8a8f9e9\', \'imgs.zip\',\n                   \'44eb7d9214e1ac5b7946e237b669401866df9114fd9a3618c5a84fbffcded0b0\'),\n    \'slake_train\': (\'BoKelvin/SLAKE\', \'dataset\', \'a9083ce6c34ac3ffb17671a605962924d8a8f9e9\', \'train.json\',\n                    \'7a9fb81f9d7bb145c3d7e9bc8d3b9016ba18b6539ea5031a53bae8fd3cd991c3\'),\n    \'slake_test\': (\'BoKelvin/SLAKE\', \'dataset\', \'a9083ce6c34ac3ffb17671a605962924d8a8f9e9\', \'test.json\',\n                   \'6be8f7b4c5a46cdbc713a5210a25b6ed5aa1fd1574c83cefb4f998131f17c2c3\'),\n    \'RN50\': (\'sarahESL/PubMedCLIP\', \'model\', \'c5885a5c0ba8ac8d86341e57c8466e49783874c1\', \'PubMedCLIP_RN50.pth\',\n             \'bb8b2bfd7f6e38fb7fb1861d969e4df862776f605aede4ecb200d2e72f9d4c6a\'),\n    \'ViT\': (\'sarahESL/PubMedCLIP\', \'model\', \'c5885a5c0ba8ac8d86341e57c8466e49783874c1\', \'PubMedCLIP_ViT32.pth\',\n            \'56d244ea2e547b59fd0357eb17d481240eeb21590e5bae2c91ea0a55de82d9c6\'),\n}\n\n# Expert-CFG Table 1 PubMedCLIP rows = PubMedCLIP paper Table 1 (QCR + AE): [open, closed, overall].\nRUNS = {\n    \'rad\': {\'encoder\': \'RN50\', \'config\': \'qcr_pubmedclipRN50_ae_rad_16batchsize_withtfidf_nondeterministic.yaml\',\n            \'paper\': [60.10, 80.00, 72.10], \'counts\': {\'train\': 3064, \'test\': 451}},\n    \'slake\': {\'encoder\': \'ViT\', \'config\': \'qcr_pubmedclipViT_ae_slake_16batchsize_withtfidf_nondeterministic.yaml\',\n              \'paper\': [78.40, 82.50, 80.10], \'counts\': {\'train\': 4919, \'test\': 1061}},\n}\n\n# The original QCR authors\' repository ships the prepared VQA-RAD bundle (data/). PubMedCLIP\'s\n# QCR code at QCR_COMMIT hard-codes its label space (487 = 56 closed + 431 open, open labels\n# indexed locally) and its type classifier was trained on that 1177-word dictionary; the\n# PubMedCLIP create_label.py/create_dictionary.py would instead produce 64+483 labels and\n# 1259 words, which the code cannot load. Train/test JSON, imgid2idx, auto-encoder and\n# images128 are byte-identical to the MEVF bundle on HF (aiozai/MEVF data_RAD.zip).\nORIGINAL_QCR_REPO = \'https://github.com/Awenbocc/med-vqa.git\'\nORIGINAL_QCR_COMMIT = \'0cca6811e38cf54aff6a7cce3442296d07875e64\'\nRAD_SHA256 = {\n    \'trainset.json\': \'3b0567673ae224b6489ca1bc4ce05cda642223d5cf4f17db8fbf7010c872e00f\',\n    \'testset.json\': \'3d32a047592cb680e18054030b38b1a44d6b17105fc62d3088c8655d56a00ced\',\n    \'imgid2idx.json\': \'4ee2f8ebc4adf3700c10307c0b3daac3edafaf08717b4d20eef423228e316257\',\n    \'dictionary.pkl\': \'35a01367a5d5a4417d6f99c045bf8f3ccca51bb48da0e37e1aae38590cc40504\',\n    \'glove6b_init_300d.npy\': \'7b70281a90e0b141d4655481718a99f9d3f09da23d2f10608e874173c1057bf1\',\n    \'embed_tfidf_weights.pkl\': \'0bb3cbb06714f8987886abffe69e0d38bf312ce7b9406b2b804915a568fcfd27\',\n    \'images128x128.pkl\': \'136cffd2c2222993c9cc7035ded0fa11d7ce1cae37ab0baaa951f85a002c6e6f\',\n    \'pretrained_ae.pth\': \'f2fb0fdc6d014fb49b85fcd2029902e14db9b42d21f3718d1a07d7082960bd05\',\n    \'cache/close_label2ans.pkl\': \'b50f9d24cb6a54599ffda7b371f3f42419be42aa5c58f9069d531cf7ffe9d863\',\n    \'cache/open_label2ans.pkl\': \'965340e4b4e9ad152e7d055987f25cafe630666d70b6fee795ccc526d28a1f4a\',\n    \'cache/trainval_ans2label.pkl\': \'daaebd434b2d1e06d79aebf24d750adca76dcb1e4548ada83a16aee0848b6b77\',\n    \'cache/trainval_label2ans.pkl\': \'22bd133cc95f2d2c25973db5e463e0c3a603c84ea4404bec1020299b8d587e83\',\n    \'cache/train_openclose_target.pkl\': \'2b99f7ade3d38d0fb5da77a4af65f8afb505f1afb5657e2b23623126c602edf5\',\n    \'cache/test_openclose_target.pkl\': \'4df33f0095ea260d91b6525d3318281c8313cebf28f0dd5c56d363635342c09b\',\n}\n\n# Inserted into lib/utils/utils.py. Globals were audited with pickletools on the pinned file.\nTFIDF_LOADER = \'\'\'string_classes = (str, bytes)\nimport pickle as _pickle\nimport types as _types\ntry:  # referenced by the PyTorch 0.4 pickle of embed_tfidf_weights.pkl\n    import torch.nn.backends.thnn\nexcept ImportError:\n    import sys as _sys\n    _thnn = _types.ModuleType(\'torch.nn.backends.thnn\')\n    _thnn._get_thnn_function_backend = lambda: None\n    _sys.modules[\'torch.nn.backends.thnn\'] = _thnn\n_TFIDF_GLOBALS = {(\'collections\', \'OrderedDict\'), (\'language_model\', \'WordEmbedding\'),\n                  (\'language.language_model\', \'WordEmbedding\'),  # file written by QCR itself (SLAKE)\n                  (\'__builtin__\', \'set\'), (\'builtins\', \'set\'),\n                  (\'torch\', \'FloatStorage\'), (\'torch._utils\', \'_rebuild_parameter\'),\n                  (\'torch._utils\', \'_rebuild_tensor_v2\'), (\'torch.nn.backends.thnn\', \'_get_thnn_function_backend\'),\n                  (\'torch.nn.modules.dropout\', \'Dropout\'), (\'torch.nn.modules.sparse\', \'Embedding\')}\nclass _AllowlistUnpickler(_pickle.Unpickler):\n    def find_class(self, module, name):\n        if (module, name) not in _TFIDF_GLOBALS:\n            raise _pickle.UnpicklingError(\'Blocked global in embed_tfidf_weights.pkl: %s.%s\' % (module, name))\n        return super().find_class(module, name)\n_tfidf_pickle = _types.ModuleType(\'qcr_allowlist_pickle\')\n_tfidf_pickle.Unpickler = _AllowlistUnpickler\n_tfidf_pickle.load = lambda f, **kw: _AllowlistUnpickler(f, **kw).load()\n\'\'\'\n\n# (file, old, new, reason). Each `old` must occur exactly once in the pinned commit.\nPATCHES = [\n    (\'lib/utils/utils.py\', \'from torch._six import string_classes\\n\', TFIDF_LOADER,\n     \'torch._six was removed in PyTorch 2; also defines the allowlisted loader used below\'),\n    (\'lib/utils/utils.py\', \'collections.Mapping)\', \'collections.abc.Mapping)\', \'Python 3.10 removed collections.Mapping\'),\n    (\'lib/utils/utils.py\', \'collections.Sequence)\', \'collections.abc.Sequence)\', \'Python 3.10 removed collections.Sequence\'),\n    (\'lib/utils/utils.py\', \'w_emb = torch.load(f)\\n\',\n     \'w_emb = torch.load(f, weights_only=False, pickle_module=_tfidf_pickle)\\n\',\n     \'embed_tfidf_weights.pkl is a PyTorch-0.4 pickled nn.Module; PyTorch>=2.6 defaults to weights_only=True. \'\n     \'Load it with an unpickler that only allows the audited classes\'),\n    (\'lib/utils/utils.py\', \'torch.sparse.FloatTensor(torch.LongTensor(inds), torch.FloatTensor(vals))\',\n     \'torch.sparse_coo_tensor(torch.LongTensor(inds), torch.FloatTensor(vals))\', \'legacy sparse constructor\'),\n    (\'lib/utils/create_label.py\', \'qa_pairs = train_qa_pairs.append(val_qa_pairs)\\n    qa_pairs[\\\'answer\\\'] = qa_pairs[\\\'answer\\\'].apply(lambda x: str(x))\\n\\n    for id\',\n     \'qa_pairs = pd.concat([train_qa_pairs, val_qa_pairs])\\n    qa_pairs[\\\'answer\\\'] = qa_pairs[\\\'answer\\\'].apply(lambda x: str(x))\\n\\n    for id\',\n     \'DataFrame.append was removed in pandas 2\'),\n    (\'lib/utils/create_label.py\', \'qa_pairs = train_qa_pairs.append(val_qa_pairs)\\n    qa_pairs[\\\'answer\\\'] = qa_pairs[\\\'answer\\\'].apply(lambda x: str(x))\\n    qa_pairs_open\',\n     \'qa_pairs = pd.concat([train_qa_pairs, val_qa_pairs])\\n    qa_pairs[\\\'answer\\\'] = qa_pairs[\\\'answer\\\'].apply(lambda x: str(x))\\n    qa_pairs_open\',\n     \'DataFrame.append was removed in pandas 2\'),\n    (\'lib/utils/create_label.py\', \'df = train_qa_pairs.append(val_qa_pairs)\', \'df = pd.concat([train_qa_pairs, val_qa_pairs])\',\n     \'DataFrame.append was removed in pandas 2\'),\n    (\'lib/utils/create_img2idx.py\', \'df = train_en.append(val_en)\', \'df = pd.concat([train_en, val_en])\',\n     \'DataFrame.append was removed in pandas 2\'),\n    (\'lib/BAN/multi_level_model.py\',\n     "            self.clip, _ = clip.load(cfg.TRAIN.VISION.CLIP_VISION_ENCODER, jit=False)\\n"\n     "            if not cfg.TRAIN.VISION.CLIP_ORG:\\n"\n     "                checkpoint = torch.load(cfg.TRAIN.VISION.CLIP_PATH)\\n"\n     "                self.clip.load_state_dict(checkpoint[\'state_dict\'])\\n",\n     "            if not cfg.TRAIN.VISION.CLIP_ORG:\\n"\n     "                checkpoint = torch.load(cfg.TRAIN.VISION.CLIP_PATH, map_location=\'cpu\')\\n"\n     "                # Same as clip.load(name) + load_state_dict: build_model() casts to fp16 and the\\n"\n     "                # PubMedCLIP weights are copied into those fp16 parameters before .float().\\n"\n     "                self.clip = clip.model.build_model(checkpoint[\'state_dict\']).to(device)\\n"\n     "                self.clip.load_state_dict(checkpoint[\'state_dict\'])\\n"\n     "            else:\\n"\n     "                self.clip, _ = clip.load(cfg.TRAIN.VISION.CLIP_VISION_ENCODER, jit=False)\\n",\n     \'clip.load(name) first downloads OpenAI weights (openaipublic.azureedge.net CDN) that are immediately \'\n     \'overwritten; build the identical architecture from the PubMedCLIP state dict instead\'),\n    (\'main/classifier.py\', \'nn.Dropout(cfg.TRAIN.DROPOUT, inplace=True)\', \'nn.Dropout(cfg.TRAIN.DROPOUT, inplace=False)\',\n     \'current autograd rejects in-place dropout on the saved ReLU output; out-of-place dropout gives the same \'\n     \'forward values and gradients (dropped units get zero gradient either way)\'),\n    (\'main/train.py\',\n     "                utils.save_model(model_path, model, best_epoch, eval_score, open_score, close_score, optim)\\n",\n     "                utils.save_model(model_path, model, best_epoch, eval_score, open_score, close_score, optim)\\n"\n     "                if os.environ.get(\'QCR_KEEP_ALL_BEST\', \'0\') != \'1\':\\n"\n     "                    for old in os.listdir(ckpt_path):\\n"\n     "                        if old.endswith(\'_best.pth\') and old != os.path.basename(model_path):\\n"\n     "                            os.remove(os.path.join(ckpt_path, old))\\n",\n     \'each improvement saves a ~1 GB checkpoint (CLIP + optimizer); keep only the newest one so 200 epochs fit \'\n     \'on Colab disk (file handling only; QCR_KEEP_ALL_BEST=1 restores the original behaviour)\'),\n    (\'main/main.py\', \'    torch.autograd.set_detect_anomaly(True)\\n\',\n     \'    torch.autograd.set_detect_anomaly(os.environ.get("QCR_DETECT_ANOMALY", "0") == "1")\\n\',\n     \'anomaly detection is a debugging aid (no numerical effect) that slows training several times; \'\n     \'set QCR_DETECT_ANOMALY=1 to restore it\'),\n]\n\n\ndef sha256(path, chunk=1 << 22):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(chunk), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef run(cmd, cwd=None, env=None):\n    import os\n    print(\'$\', \' \'.join(map(str, cmd)), flush=True)\n    # The authors\' scripts open GloVe/JSON with the platform default encoding.\n    env = dict(env or os.environ, PYTHONUTF8=\'1\', PYTHONUNBUFFERED=\'1\')\n    # Stream through Python so the output is visible in a notebook cell.\n    with subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=env, stdout=subprocess.PIPE,\n                          stderr=subprocess.STDOUT, text=True, encoding=\'utf-8\', errors=\'replace\') as proc:\n        for line in proc.stdout:\n            print(line, end=\'\', flush=True)\n    if proc.returncode:\n        raise subprocess.CalledProcessError(proc.returncode, cmd)\n\n\ndef fetch(key, cache_dir):\n    from huggingface_hub import hf_hub_download\n    repo, repo_type, revision, filename, expected = ASSETS[key]\n    path = Path(hf_hub_download(repo, filename, repo_type=repo_type, revision=revision, cache_dir=str(cache_dir)))\n    if expected and sha256(path) != expected:\n        raise ValueError(f\'SHA-256 mismatch for {repo}/{filename}@{revision}\')\n    return path\n\n\ndef safe_extract(zip_path, target):\n    target = Path(target)\n    with zipfile.ZipFile(zip_path) as archive:\n        for member in archive.infolist():\n            name = PurePosixPath(member.filename)\n            if name.is_absolute() or \'..\' in name.parts or \'\\\\\' in member.filename:\n                raise ValueError(\'Unsafe archive member: \' + member.filename)\n        archive.extractall(target)\n\n\ndef setup_code(work_dir, source=None):\n    """Clone the pinned official repo (or copy a local clone) and apply PATCHES."""\n    work_dir = Path(work_dir)\n    repo_root = work_dir / \'PubMedCLIP\'\n    if not repo_root.exists():\n        if source:\n            shutil.copytree(source, repo_root, ignore=shutil.ignore_patterns(\'.git\'))\n        else:\n            run([\'git\', \'clone\', \'--quiet\', QCR_REPO, repo_root])\n            run([\'git\', \'checkout\', \'--quiet\', QCR_COMMIT], cwd=repo_root)\n    if (repo_root / \'.git\').exists():\n        head = subprocess.run([\'git\', \'rev-parse\', \'HEAD\'], cwd=repo_root, capture_output=True, text=True, check=True).stdout.strip()\n        if head != QCR_COMMIT:\n            raise ValueError(f\'Expected QCR commit {QCR_COMMIT}, found {head}\')\n    qcr = repo_root / \'QCR_PubMedCLIP\'\n    applied = []\n    for rel, old, new, reason in PATCHES:\n        path = qcr / rel\n        text = path.read_text(encoding=\'utf-8\')\n        if text.count(old) == 1:\n            path.write_text(text.replace(old, new), encoding=\'utf-8\')\n        elif not (text.count(old) == 0 and new in text):\n            raise ValueError(f\'Patch target not found exactly once in {rel}: {old[:60]!r}\')\n        applied.append({\'file\': rel, \'reason\': reason})\n    return qcr, applied\n\n\ndef _check_counts(dataset, data_dir):\n    if dataset == \'rad\':\n        train = json.loads((data_dir / \'trainset.json\').read_text(encoding=\'utf-8\'))\n        test = json.loads((data_dir / \'testset.json\').read_text(encoding=\'utf-8\'))\n    else:\n        train = [r for r in json.loads((data_dir / \'train.json\').read_text(encoding=\'utf-8\')) if r[\'q_lang\'] == \'en\']\n        test = [r for r in json.loads((data_dir / \'test.json\').read_text(encoding=\'utf-8\')) if r[\'q_lang\'] == \'en\']\n    counts = {\'train\': len(train), \'test\': len(test)}\n    if counts != RUNS[dataset][\'counts\']:\n        raise ValueError(f\'{dataset}: expected {RUNS[dataset]["counts"]}, got {counts}\')\n    return counts\n\n\ndef _clone_original_qcr(work_dir):\n    repo = Path(work_dir) / \'med-vqa\'\n    if not repo.exists():\n        run([\'git\', \'clone\', \'--quiet\', ORIGINAL_QCR_REPO, repo])\n    run([\'git\', \'checkout\', \'--quiet\', ORIGINAL_QCR_COMMIT], cwd=repo)\n    return repo\n\n\ndef _extract_member(zip_path, member, target):\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(zip_path) as archive, archive.open(member) as src, open(target, \'wb\') as dst:\n        shutil.copyfileobj(src, dst, 1 << 22)\n\n\ndef prepare_data(qcr, dataset, cache_dir):\n    """Create data/data_rad or data/data_slake as the QCR code at QCR_COMMIT expects."""\n    qcr, cache_dir = Path(qcr), Path(cache_dir)\n    data_root = qcr / \'data\'\n    rad_dir, slake_dir = data_root / \'data_rad\', data_root / \'data_slake\'\n    utils_dir = qcr / \'lib\' / \'utils\'\n    py = sys.executable\n    marker = data_root / f\'.{dataset}_ready\'\n    if marker.exists():\n        return json.loads(marker.read_text(encoding=\'utf-8\'))\n    if dataset == \'rad\':\n        source = _clone_original_qcr(qcr.parent.parent) / \'data\'\n        if rad_dir.exists():\n            shutil.rmtree(rad_dir)\n        shutil.copytree(source, rad_dir, ignore=shutil.ignore_patterns(\'*.ipynb\', \'.ipynb_checkpoints\'))\n        for rel, expected in RAD_SHA256.items():\n            if sha256(rad_dir / rel) != expected:\n                raise ValueError(f\'Original QCR RAD file changed: {rel}\')\n        counts = _check_counts(\'rad\', rad_dir)\n        # The only input missing from the bundle: CLIP images, made with PubMedCLIP\'s own script\n        # exactly as in lib/utils/run.sh (250x250, 3 channels).\n        run([py, \'create_resized_images.py\', rad_dir / \'imgid2idx.json\', rad_dir / \'images\',\n             250, rad_dir / \'images250x250.pkl\', 3], cwd=utils_dir)\n        assets = {\'original_qcr\': [ORIGINAL_QCR_REPO, ORIGINAL_QCR_COMMIT, \'data/\']}\n    elif dataset == \'slake\':\n        # lib/utils/run.sh, SLAKE part. GloVe 6B-300d and the pretrained auto-encoder come from the\n        # MEVF bundle on HF (the same auto-encoder file as the RAD bundle; QCR ships no SLAKE one).\n        mevf = fetch(\'mevf_rad\', cache_dir)\n        glove = data_root / \'glove.6B\' / \'glove.6B.300d.txt\'\n        if not glove.exists():\n            _extract_member(mevf, \'glove/glove.6B.300d.txt\', glove)\n        slake_dir.mkdir(parents=True, exist_ok=True)\n        _extract_member(mevf, \'pretrained_ae.pth\', slake_dir / \'pretrained_ae.pth\')\n        if sha256(slake_dir / \'pretrained_ae.pth\') != RAD_SHA256[\'pretrained_ae.pth\']:\n            raise ValueError(\'Unexpected auto-encoder checkpoint\')\n        for key, name in ((\'slake_train\', \'train.json\'), (\'slake_test\', \'test.json\')):\n            shutil.copyfile(fetch(key, cache_dir), slake_dir / name)\n        if not (slake_dir / \'imgs\').exists():\n            safe_extract(fetch(\'slake_imgs\', cache_dir), slake_dir)\n        counts = _check_counts(\'slake\', slake_dir)\n        run([py, \'create_dictionary.py\', slake_dir.as_posix()], cwd=utils_dir)\n        run([py, \'create_label.py\', slake_dir.as_posix() + \'/\'], cwd=utils_dir)\n        run([py, \'create_img2idx.py\', slake_dir / \'train.json\', slake_dir / \'test.json\', slake_dir / \'imgid2idx.json\'], cwd=utils_dir)\n        for size, channels in ((128, 1), (250, 3)):\n            run([py, \'create_resized_images.py\', slake_dir / \'imgid2idx.json\', slake_dir / \'imgs\',\n                 size, slake_dir / f\'images{size}x{size}.pkl\', channels], cwd=utils_dir)\n        assets = {k: list(ASSETS[k][:4]) for k in (\'mevf_rad\', \'slake_imgs\', \'slake_train\', \'slake_test\')}\n    else:\n        raise ValueError(\'dataset must be rad or slake\')\n    info = {\'dataset\': dataset, \'counts\': counts, \'assets\': assets}\n    marker.write_text(json.dumps(info, indent=2), encoding=\'utf-8\')\n    return info\n\n\ndef make_config(qcr, dataset, clip_path, run_index, n_epoch=None):\n    """Copy the authors\' config; only CLIP_PATH, the run NAME and (quick checks only) N_EPOCH change."""\n    qcr = Path(qcr)\n    spec = RUNS[dataset]\n    text = (qcr / \'configs\' / spec[\'config\']).read_text(encoding=\'utf-8\')\n    text, n = re.subn(r\'CLIP_PATH: ".*"\', f\'CLIP_PATH: "{Path(clip_path).as_posix()}"\', text)\n    assert n == 1\n    suffix = f\'.run{run_index}\' + (f\'.epochs{n_epoch}\' if n_epoch else \'\')\n    text, n = re.subn(r"^NAME: \'(.*)\'", lambda m: f"NAME: \'{m.group(1)}{suffix}\'", text, flags=re.M)\n    assert n == 1\n    if n_epoch:\n        text, n = re.subn(r\'N_EPOCH: \\d+\', f\'N_EPOCH: {int(n_epoch)}\', text)\n        assert n == 1\n    out = qcr / \'configs\' / f\'repro_{dataset}{suffix}.yaml\'\n    out.write_text(text, encoding=\'utf-8\')\n    return out\n\n\ndef config_value(config_path, key):\n    match = re.search(rf\'^\\s*{key}:\\s*[\\\'"]?([^\\\'"\\n#]+)\', Path(config_path).read_text(encoding=\'utf-8\'), re.M)\n    return match.group(1).strip()\n\n\nVAL_RE = re.compile(r\'\\[Validate\\] Val_Acc:([\\d.]+)%\\s*\\|\\s*Open_ACC:([\\d.]+)%\\s*\\|\\s*Close_ACC:([\\d.]+)%\')\n\n\ndef parse_log(log_path):\n    """Per-epoch test accuracy from medVQA.log; best epoch selected exactly as train.py does."""\n    epochs = [{\'epoch\': i, \'overall\': float(a), \'open\': float(o), \'closed\': float(c)}\n              for i, (a, o, c) in enumerate(VAL_RE.findall(Path(log_path).read_text(encoding=\'utf-8\', errors=\'replace\')))]\n    best = None\n    for row in epochs:  # train.py keeps the first epoch with a strictly higher overall accuracy\n        if row[\'overall\'] > (best[\'overall\'] if best else 0):\n            best = row\n    return {\'epochs_logged\': len(epochs), \'best\': best, \'last\': epochs[-1] if epochs else None}\n\n\ndef train(qcr, dataset, clip_path, run_index=0, n_epoch=None, anomaly=False, keep_all_best=False):\n    """Run the authors\' main/main.py with their config (see make_config) and parse its log."""\n    import os\n    qcr = Path(qcr)\n    config = make_config(qcr, dataset, clip_path, run_index, n_epoch)\n    env = dict(os.environ, QCR_DETECT_ANOMALY=\'1\' if anomaly else \'0\', QCR_KEEP_ALL_BEST=\'1\' if keep_all_best else \'0\')\n    run([sys.executable, \'main/main.py\', \'--cfg\', config.relative_to(qcr).as_posix()], cwd=qcr, env=env)\n    out = qcr / config_value(config, \'OUTPUT_DIR\') / config_value(config, \'NAME\')\n    result = parse_log(out / \'medVQA.log\')\n    expected_epochs = int(config_value(config, \'N_EPOCH\'))\n    result.update(dataset=dataset, encoder=RUNS[dataset][\'encoder\'], run=run_index, config=config.name,\n                  full_protocol=n_epoch is None and result[\'epochs_logged\'] == expected_epochs,\n                  paper_open_closed_overall=RUNS[dataset][\'paper\'], output_dir=str(out))\n    (out / \'result.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    return result\n\n\ndef summarize(results):\n    """Mean/std over runs of each run\'s best-epoch test accuracy (the authors\' protocol) and last epoch."""\n    import statistics\n    table = {}\n    for dataset in RUNS:\n        rows = [r for r in results if r[\'dataset\'] == dataset and r[\'best\']]\n        if not rows:\n            continue\n        entry = {\'runs\': len(rows), \'full_protocol\': all(r[\'full_protocol\'] for r in rows),\n                 \'paper_open_closed_overall\': RUNS[dataset][\'paper\']}\n        for key in (\'open\', \'closed\', \'overall\'):\n            values = [r[\'best\'][key] for r in rows]\n            entry[key] = statistics.mean(values)\n            entry[key + \'_std\'] = statistics.stdev(values) if len(values) > 1 else 0.0\n            entry[\'last_epoch_\' + key] = statistics.mean(r[\'last\'][key] for r in rows)\n        table[dataset] = entry\n    return table\n', encoding='utf-8')
(CODE_ROOT / 'biomedclip_meter.py').write_text('"""BiomedCLIP + METER-style VQA on VQA-RAD / SLAKE: an APPROXIMATION of the BiomedCLIP paper row.\n\nThe BiomedCLIP paper (Table 11) fine-tunes "ViT-B/16-448-PMB/256" with METER but releases neither\nthe 448 checkpoint nor VQA code/hyper-parameters. This module therefore:\n  * uses the public BiomedCLIP checkpoint (ViT-B/16-224 + PubMedBERT), pinned revision;\n  * re-implements the METER co-attention fusion (zdou0830/METER: 6 BertCrossLayers per modality,\n    CLS poolers, VQA head) with randomly initialised fusion, full fine-tuning of both encoders;\n  * fills every unstated setting from M3AE (zhjohnchan/M3AE, a METER-based medical VQA code base):\n    AdamW lr 5e-6, x5 cross-modal, x50 head, wd 0.01, 10% warm-up + linear decay, BCE x labels,\n    answer set = all normalised answers of every split, max 32 question tokens,\n    VQA-RAD batch 64 / 50 epochs, SLAKE batch 32 / 15 epochs;\n  * evaluates like QCR/M3AE (test accuracy every epoch, best epoch reported) and also records the\n    last epoch and, for SLAKE, the epoch chosen on the official validation split.\nData: the same VQA-RAD 3064/451 and SLAKE English files as the PubMedCLIP QCR reproduction (qcr_port).\n"""\nimport base64\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport random\nimport re\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path\n\nimport qcr_port\n\nBIOMEDCLIP = (\'microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224\', \'9f341de24bfb00180f1b847274256e9b65a3a32e\',\n              \'open_clip_pytorch_model.bin\', \'52cc993c5c5ff962bd0c60931874bc001e7e9b41666a385530f4a036294576be\')\nTEXT_CONFIG = (\'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract\', \'d673b8835373c6fa116d6d8006b33d48734e305d\')\nSLAKE_VALIDATION = (\'BoKelvin/SLAKE\', \'dataset\', \'a9083ce6c34ac3ffb17671a605962924d8a8f9e9\', \'validation.json\',\n                    \'32b016440b0c3be11056a78a18eeab46333268407fbb6e6b32f9f4c2debc50f6\')\nqcr_port.ASSETS.setdefault(\'slake_validation\', SLAKE_VALIDATION)\nCLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)\n\n# BiomedCLIP paper Table 11 (METER row) = Expert-CFG Table 1 BiomedCLIP row: [open, closed, overall].\nPAPER = {\'rad\': [67.60, 79.80, 75.20], \'slake\': [82.50, 89.70, 85.40]}\nCOUNTS = {\'rad\': {\'train\': 3064, \'test\': 451}, \'slake\': {\'train\': 4919, \'validation\': 1053, \'test\': 1061}}\n# M3AE task_finetune_vqa_vqa_rad / task_finetune_vqa_slack (+ METER defaults for the rest).\nDEFAULTS = {\n    \'rad\': dict(batch_size=64, epochs=50),\n    \'slake\': dict(batch_size=32, epochs=15),\n    \'common\': dict(learning_rate=5e-6, lr_mult_cross_modal=5, lr_mult_head=50, weight_decay=0.01, warmup=0.1,\n                   betas=(0.9, 0.98), eps=1e-8, max_text_len=32, image_size=224, num_top_layer=6, hidden_size=768,\n                   num_heads=12, mlp_ratio=4, drop_rate=0.1, micro_batch=16, seed=0, amp=True, grad_checkpointing=True, num_workers=2),\n}\n\n# ---- VQA answer normalisation (M3AE prepro/glossary.py normalize_word; same tables as QCR create_label.py)\nCONTRACTIONS = {\'aint\': "ain\'t", \'arent\': "aren\'t", \'cant\': "can\'t", \'couldve\': "could\'ve", \'couldnt\': "couldn\'t", "couldn\'tve": "couldn\'t\'ve", "couldnt\'ve": "couldn\'t\'ve", \'didnt\': "didn\'t", \'doesnt\': "doesn\'t", \'dont\': "don\'t", \'hadnt\': "hadn\'t", "hadnt\'ve": "hadn\'t\'ve", "hadn\'tve": "hadn\'t\'ve", \'hasnt\': "hasn\'t", \'havent\': "haven\'t", \'hed\': "he\'d", "hed\'ve": "he\'d\'ve", "he\'dve": "he\'d\'ve", \'hes\': "he\'s", \'howd\': "how\'d", \'howll\': "how\'ll", \'hows\': "how\'s", "Id\'ve": "I\'d\'ve", "I\'dve": "I\'d\'ve", \'Im\': "I\'m", \'Ive\': "I\'ve", \'isnt\': "isn\'t", \'itd\': "it\'d", "itd\'ve": "it\'d\'ve", "it\'dve": "it\'d\'ve", \'itll\': "it\'ll", "let\'s": "let\'s", \'maam\': "ma\'am", \'mightnt\': "mightn\'t", "mightnt\'ve": "mightn\'t\'ve", "mightn\'tve": "mightn\'t\'ve", \'mightve\': "might\'ve", \'mustnt\': "mustn\'t", \'mustve\': "must\'ve", \'neednt\': "needn\'t", \'notve\': "not\'ve", \'oclock\': "o\'clock", \'oughtnt\': "oughtn\'t", "ow\'s\'at": "\'ow\'s\'at", "\'ows\'at": "\'ow\'s\'at", "\'ow\'sat": "\'ow\'s\'at", \'shant\': "shan\'t", "shed\'ve": "she\'d\'ve", "she\'dve": "she\'d\'ve", "she\'s": "she\'s", \'shouldve\': "should\'ve", \'shouldnt\': "shouldn\'t", "shouldnt\'ve": "shouldn\'t\'ve", "shouldn\'tve": "shouldn\'t\'ve", "somebody\'d": \'somebodyd\', "somebodyd\'ve": "somebody\'d\'ve", "somebody\'dve": "somebody\'d\'ve", \'somebodyll\': "somebody\'ll", \'somebodys\': "somebody\'s", \'someoned\': "someone\'d", "someoned\'ve": "someone\'d\'ve", "someone\'dve": "someone\'d\'ve", \'someonell\': "someone\'ll", \'someones\': "someone\'s", \'somethingd\': "something\'d", "somethingd\'ve": "something\'d\'ve", "something\'dve": "something\'d\'ve", \'somethingll\': "something\'ll", \'thats\': "that\'s", \'thered\': "there\'d", "thered\'ve": "there\'d\'ve", "there\'dve": "there\'d\'ve", \'therere\': "there\'re", \'theres\': "there\'s", \'theyd\': "they\'d", "theyd\'ve": "they\'d\'ve", "they\'dve": "they\'d\'ve", \'theyll\': "they\'ll", \'theyre\': "they\'re", \'theyve\': "they\'ve", \'twas\': "\'twas", \'wasnt\': "wasn\'t", "wed\'ve": "we\'d\'ve", "we\'dve": "we\'d\'ve", \'weve\': "we\'ve", \'werent\': "weren\'t", \'whatll\': "what\'ll", \'whatre\': "what\'re", \'whats\': "what\'s", \'whatve\': "what\'ve", \'whens\': "when\'s", \'whered\': "where\'d", \'wheres\': "where\'s", \'whereve\': "where\'ve", \'whod\': "who\'d", "whod\'ve": "who\'d\'ve", "who\'dve": "who\'d\'ve", \'wholl\': "who\'ll", \'whos\': "who\'s", \'whove\': "who\'ve", \'whyll\': "why\'ll", \'whyre\': "why\'re", \'whys\': "why\'s", \'wont\': "won\'t", \'wouldve\': "would\'ve", \'wouldnt\': "wouldn\'t", "wouldnt\'ve": "wouldn\'t\'ve", "wouldn\'tve": "wouldn\'t\'ve", \'yall\': "y\'all", "yall\'ll": "y\'all\'ll", "y\'allll": "y\'all\'ll", "yall\'d\'ve": "y\'all\'d\'ve", "y\'alld\'ve": "y\'all\'d\'ve", "y\'all\'dve": "y\'all\'d\'ve", \'youd\': "you\'d", "youd\'ve": "you\'d\'ve", "you\'dve": "you\'d\'ve", \'youll\': "you\'ll", \'youre\': "you\'re", \'youve\': "you\'ve"}\nMANUAL_MAP = {\'none\': \'0\', \'zero\': \'0\', \'one\': \'1\', \'two\': \'2\', \'three\': \'3\', \'four\': \'4\', \'five\': \'5\', \'six\': \'6\', \'seven\': \'7\', \'eight\': \'8\', \'nine\': \'9\', \'ten\': \'10\'}\nARTICLES = [\'a\', \'an\', \'the\']\nPUNCT = [\';\', \'/\', \'[\', \']\', \'"\', \'{\', \'}\', \'(\', \')\', \'=\', \'+\', \'\\\\\', \'_\', \'-\', \'>\', \'<\', \'@\', \'`\', \',\', \'?\', \'!\']\n_PERIOD_STRIP = re.compile(r\'(?!<=\\d)(\\.)(?!\\d)\')\n_COMMA_STRIP = re.compile(r\'(\\d)(\\,)(\\d)\')\n\n\ndef normalize_answer(token):\n    token = str(token).lower()\n    _token = token\n    for p in PUNCT:\n        if (p + \' \' in token or \' \' + p in token) or (re.search(_COMMA_STRIP, token) is not None):\n            _token = _token.replace(p, \'\')\n        else:\n            _token = _token.replace(p, \' \')\n    token = _PERIOD_STRIP.sub(\'\', _token, re.UNICODE)\n    words = [MANUAL_MAP.get(w, w) for w in token.lower().split()]\n    words = [CONTRACTIONS.get(w, w) for w in words if w not in ARTICLES]\n    return \' \'.join(words).replace(\',\', \'\')\n\n\n# ---- data -----------------------------------------------------------------------------------------\ndef load_data(dataset, work_dir, cache_dir):\n    """Rows with original question/answer/answer_type; the files are those of the QCR reproduction."""\n    work_dir, cache_dir = Path(work_dir), Path(cache_dir)\n    splits = {}\n    if dataset == \'rad\':\n        repo = qcr_port._clone_original_qcr(work_dir)\n        data = repo / \'data\'\n        for name in (\'trainset.json\', \'testset.json\'):\n            if qcr_port.sha256(data / name) != qcr_port.RAD_SHA256[name]:\n                raise ValueError(\'VQA-RAD split file changed: \' + name)\n        for split in (\'train\', \'test\'):\n            rows = json.loads((data / f\'{split}set.json\').read_text(encoding=\'utf-8\'))\n            splits[split] = [dict(qid=r[\'qid\'], question=r[\'question\'], answer=str(r[\'answer\']),\n                                  answer_type=r[\'answer_type\'], image=str(data / \'images\' / r[\'image_name\'])) for r in rows]\n    elif dataset == \'slake\':\n        root = work_dir / \'slake\'\n        if not (root / \'imgs\').exists():\n            root.mkdir(parents=True, exist_ok=True)\n            qcr_port.safe_extract(qcr_port.fetch(\'slake_imgs\', cache_dir), root)\n        for split, key in ((\'train\', \'slake_train\'), (\'validation\', \'slake_validation\'), (\'test\', \'slake_test\')):\n            rows = json.loads(Path(qcr_port.fetch(key, cache_dir)).read_text(encoding=\'utf-8\'))\n            splits[split] = [dict(qid=r[\'qid\'], question=r[\'question\'], answer=str(r[\'answer\']),\n                                  answer_type=r[\'answer_type\'], image=str(root / \'imgs\' / r[\'img_name\']))\n                             for r in rows if r[\'q_lang\'] == \'en\']\n    else:\n        raise ValueError(\'dataset must be rad or slake\')\n    counts = {k: len(v) for k, v in splits.items()}\n    if counts != COUNTS[dataset]:\n        raise ValueError(f\'{dataset}: expected {COUNTS[dataset]}, got {counts}\')\n    for rows in splits.values():\n        for r in rows:\n            kind = r[\'answer_type\'].strip().upper()\n            if kind not in (\'OPEN\', \'CLOSED\'):\n                raise ValueError(f\'Bad answer_type {r["answer_type"]!r}\')\n            r[\'answer_type\'], r[\'label_text\'] = kind, normalize_answer(r[\'answer\'])\n            if not Path(r[\'image\']).exists():\n                raise FileNotFoundError(r[\'image\'])\n    # M3AE make_arrow_vqa: label set = every normalised answer of every split, in first-seen order.\n    labels = list(dict.fromkeys(r[\'label_text\'] for split in splits for r in splits[split]))\n    return splits, labels\n\n\n# ---- model ------------------------------------------------------------------------------------------\ndef build_model(num_labels, cfg, weights_path, text_config_dir):\n    import torch\n    import timm\n    from torch import nn\n    from torch.nn import functional as F\n    from transformers import BertConfig, BertModel\n    from torch.utils.checkpoint import checkpoint\n\n    hs, heads, drop = cfg[\'hidden_size\'], cfg[\'num_heads\'], cfg[\'drop_rate\']\n\n    class Attention(nn.Module):  # METER/BERT BertAttention: MHA + dense + dropout + residual LayerNorm\n        def __init__(self):\n            super().__init__()\n            self.query, self.key, self.value = nn.Linear(hs, hs), nn.Linear(hs, hs), nn.Linear(hs, hs)\n            self.dense, self.dropout, self.LayerNorm = nn.Linear(hs, hs), nn.Dropout(drop), nn.LayerNorm(hs, eps=1e-12)\n\n        def forward(self, x, context, mask):\n            b, n, _ = x.shape\n            split = lambda t: t.view(b, -1, heads, hs // heads).transpose(1, 2)\n            out = F.scaled_dot_product_attention(split(self.query(x)), split(self.key(context)), split(self.value(context)),\n                                                 attn_mask=mask, dropout_p=drop if self.training else 0.0)\n            return self.LayerNorm(self.dropout(self.dense(out.transpose(1, 2).reshape(b, n, hs))) + x)\n\n    class BertCrossLayer(nn.Module):  # METER meter/modules/bert_model.py: self-attn -> cross-attn -> FFN\n        def __init__(self):\n            super().__init__()\n            self.attention, self.crossattention = Attention(), Attention()\n            self.intermediate = nn.Linear(hs, hs * cfg[\'mlp_ratio\'])\n            self.output, self.out_dropout = nn.Linear(hs * cfg[\'mlp_ratio\'], hs), nn.Dropout(drop)\n            self.out_norm = nn.LayerNorm(hs, eps=1e-12)\n\n        def forward(self, x, y, x_mask, y_mask):\n            x = self.crossattention(self.attention(x, x, x_mask), y, y_mask)\n            return self.out_norm(self.out_dropout(self.output(F.gelu(self.intermediate(x)))) + x)\n\n    class Pooler(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.dense = nn.Linear(hs, hs)\n\n        def forward(self, x):\n            return torch.tanh(self.dense(x[:, 0]))\n\n    class BiomedCLIPMeter(nn.Module):\n        def __init__(self):\n            super().__init__()\n            state = torch.load(weights_path, map_location=\'cpu\', weights_only=True)\n            self.vision = timm.create_model(\'vit_base_patch16_224\', pretrained=False, num_classes=0, global_pool=\'\')\n            vision_state = {k[len(\'visual.trunk.\'):]: v for k, v in state.items() if k.startswith(\'visual.trunk.\')}\n            self.vision.load_state_dict(vision_state, strict=True)\n            if cfg[\'image_size\'] != 224:  # METER-style fine-tuning at higher resolution: resample pos. embeddings\n                self.vision.set_input_size(img_size=(cfg[\'image_size\'], cfg[\'image_size\']))\n            config = BertConfig.from_pretrained(text_config_dir)\n            self.text = BertModel(config, add_pooling_layer=False)\n            text_state = {k[len(\'text.transformer.\'):]: v for k, v in state.items()\n                          if k.startswith(\'text.transformer.\') and not k.endswith(\'position_ids\')}\n            missing, unexpected = self.text.load_state_dict(text_state, strict=False)\n            if unexpected or [k for k in missing if not k.endswith(\'position_ids\')]:\n                raise ValueError(f\'BiomedCLIP text weights mismatch: missing={missing}, unexpected={unexpected}\')\n            del state\n            self.cross_modal_text_transform = nn.Linear(config.hidden_size, hs)\n            self.cross_modal_image_transform = nn.Linear(self.vision.num_features, hs)\n            self.token_type_embeddings = nn.Embedding(2, hs)\n            self.cross_modal_text_layers = nn.ModuleList([BertCrossLayer() for _ in range(cfg[\'num_top_layer\'])])\n            self.cross_modal_image_layers = nn.ModuleList([BertCrossLayer() for _ in range(cfg[\'num_top_layer\'])])\n            self.cross_modal_text_pooler, self.cross_modal_image_pooler = Pooler(), Pooler()\n            self.vqa_classifier = nn.Sequential(nn.Linear(hs * 2, hs * 2), nn.LayerNorm(hs * 2), nn.GELU(),\n                                                nn.Linear(hs * 2, num_labels))\n            for name, module in self.named_children():  # METER init_weights for the new modules\n                if name not in (\'vision\', \'text\'):\n                    module.apply(self._init)\n            if cfg[\'grad_checkpointing\']:\n                self.vision.set_grad_checkpointing(True)\n                self.text.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\': False})\n\n        @staticmethod\n        def _init(module):\n            if isinstance(module, (nn.Linear, nn.Embedding)):\n                module.weight.data.normal_(mean=0.0, std=0.02)\n            elif isinstance(module, nn.LayerNorm):\n                module.bias.data.zero_(); module.weight.data.fill_(1.0)\n            if isinstance(module, nn.Linear) and module.bias is not None:\n                module.bias.data.zero_()\n\n        def forward(self, pixel_values, input_ids, attention_mask):\n            text = self.text(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\n            image = self.vision.forward_features(pixel_values)\n            text = self.cross_modal_text_transform(text) + self.token_type_embeddings.weight[0]\n            image = self.cross_modal_image_transform(image) + self.token_type_embeddings.weight[1]\n            # Boolean SDPA masks: True = attend. Images have no padding.\n            text_mask = attention_mask[:, None, None, :].bool()\n            image_mask = None\n            for text_layer, image_layer in zip(self.cross_modal_text_layers, self.cross_modal_image_layers):\n                def step(x, y, text_layer=text_layer, image_layer=image_layer):\n                    return text_layer(x, y, text_mask, image_mask), image_layer(y, x, image_mask, text_mask)\n                if cfg[\'grad_checkpointing\'] and self.training:\n                    text, image = checkpoint(step, text, image, use_reentrant=False)\n                else:\n                    text, image = step(text, image)\n            return self.vqa_classifier(torch.cat([self.cross_modal_text_pooler(text), self.cross_modal_image_pooler(image)], -1))\n\n    return BiomedCLIPMeter()\n\n\ndef param_groups(model, cfg):\n    """METER set_schedule: x lr_mult_head for the VQA head, x lr_mult_cross_modal for \'cross_modal\', no decay on bias/norm."""\n    no_decay = (\'bias\', \'LayerNorm.bias\', \'LayerNorm.weight\', \'norm.bias\', \'norm.weight\', \'norm1.bias\', \'norm1.weight\',\n                \'norm2.bias\', \'norm2.weight\', \'out_norm.weight\', \'out_norm.bias\')\n    groups = {}\n    for name, p in model.named_parameters():\n        mult = cfg[\'lr_mult_head\'] if \'vqa_classifier\' in name else cfg[\'lr_mult_cross_modal\'] if \'cross_modal\' in name else 1\n        decay = 0.0 if any(nd in name for nd in no_decay) else cfg[\'weight_decay\']\n        groups.setdefault((mult, decay), []).append(p)\n    return [{\'params\': ps, \'lr\': cfg[\'learning_rate\'] * m, \'weight_decay\': wd} for (m, wd), ps in groups.items()]\n\n\n# ---- training ---------------------------------------------------------------------------------------\ndef fit(dataset, work_dir, cache_dir, run_index=0, overrides=None, device=\'cuda\', limit=None, micro_batch=None):\n    """Train + evaluate one run; resumable per epoch.\n\n    overrides: protocol changes (marks the run full_protocol=False). micro_batch: memory only, the effective\n    batch is kept by gradient accumulation. limit=(n_train, n_eval) subsamples for smoke tests only.\n    """\n    import numpy as np\n    import torch\n    from PIL import Image\n    from huggingface_hub import hf_hub_download, snapshot_download\n    from torch.nn import functional as F\n    from torch.utils.data import DataLoader, Dataset\n    from torchvision import transforms\n    from transformers import AutoTokenizer, get_polynomial_decay_schedule_with_warmup\n\n    cfg = config_for(dataset, overrides)\n    if micro_batch:\n        cfg[\'micro_batch\'] = micro_batch\n    full_protocol = not overrides and limit is None\n    work_dir, cache_dir = Path(work_dir), Path(cache_dir)\n    out = run_dir(work_dir, dataset, run_index, overrides, limit)\n    out.mkdir(parents=True, exist_ok=True)\n    seed = cfg[\'seed\'] + run_index\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n\n    splits, labels = load_data(dataset, work_dir, cache_dir)\n    if limit:\n        splits = {k: v[::max(1, len(v) // (limit[0] if k == \'train\' else limit[1]))][:(limit[0] if k == \'train\' else limit[1])] for k, v in splits.items()}\n    ans2label = {a: i for i, a in enumerate(labels)}\n    repo, revision, filename, expected = BIOMEDCLIP\n    weights = hf_hub_download(repo, filename, revision=revision, cache_dir=str(cache_dir))\n    if qcr_port.sha256(weights) != expected:\n        raise ValueError(\'BiomedCLIP checkpoint hash mismatch\')\n    tokenizer = AutoTokenizer.from_pretrained(repo, revision=revision, cache_dir=str(cache_dir))\n    text_config_dir = snapshot_download(TEXT_CONFIG[0], revision=TEXT_CONFIG[1], allow_patterns=[\'config.json\'], cache_dir=str(cache_dir))\n    model = build_model(len(labels), cfg, weights, text_config_dir).to(device)\n\n    size = cfg[\'image_size\']  # METER "clip" transform: resize (bicubic) + centre crop + CLIP normalisation, no augmentation\n    preprocess = transforms.Compose([transforms.Resize(size, interpolation=transforms.InterpolationMode.BICUBIC),\n                                     transforms.CenterCrop(size), transforms.ToTensor(), transforms.Normalize(CLIP_MEAN, CLIP_STD)])\n\n    class Rows(Dataset):\n        def __init__(self, rows): self.rows = rows\n        def __len__(self): return len(self.rows)\n        def __getitem__(self, i):\n            with Image.open(self.rows[i][\'image\']) as im:\n                return preprocess(im.convert(\'RGB\')), i\n\n    def collate_for(rows):\n        def collate(batch):\n            pixels, idx = zip(*batch)\n            items = [rows[i] for i in idx]\n            tok = tokenizer([r[\'question\'] for r in items], padding=\'longest\', truncation=True,\n                            max_length=cfg[\'max_text_len\'], return_tensors=\'pt\')\n            return torch.stack(pixels), tok[\'input_ids\'], tok[\'attention_mask\'], torch.tensor([ans2label[r[\'label_text\']] for r in items]), items\n        return collate\n\n    micro = min(cfg[\'micro_batch\'], cfg[\'batch_size\'])\n    accumulation = max(1, cfg[\'batch_size\'] // micro)\n    loaders = {k: DataLoader(Rows(v), batch_size=micro if k == \'train\' else 32, shuffle=k == \'train\', num_workers=cfg[\'num_workers\'],\n                             collate_fn=collate_for(v), pin_memory=device == \'cuda\', drop_last=False) for k, v in splits.items()}\n    optimizer = torch.optim.AdamW(param_groups(model, cfg), betas=tuple(cfg[\'betas\']), eps=cfg[\'eps\'])\n    steps = math.ceil(len(loaders[\'train\']) / accumulation) * cfg[\'epochs\']\n    scheduler = get_polynomial_decay_schedule_with_warmup(optimizer, int(steps * cfg[\'warmup\']), steps, lr_end=0, power=1)\n    use_amp = cfg[\'amp\'] and device == \'cuda\'\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=use_amp)\n    truncated = sum(len(tokenizer(r[\'question\'])[\'input_ids\']) > cfg[\'max_text_len\'] for v in splits.values() for r in v)\n\n    @torch.no_grad()\n    def evaluate(split):\n        model.eval()\n        hit = Counter(); total = Counter(); predictions = []\n        for pixels, ids, mask, target, items in loaders[split]:\n            with torch.autocast(device_type=\'cuda\', dtype=torch.float16, enabled=use_amp):\n                logits = model(pixels.to(device), ids.to(device), mask.to(device))\n            pred = logits.float().argmax(-1).cpu()\n            for p, t, r in zip(pred.tolist(), target.tolist(), items):\n                for group in (r[\'answer_type\'], \'ALL\'):\n                    total[group] += 1; hit[group] += int(p == t)\n                predictions.append({\'qid\': r[\'qid\'], \'answer_type\': r[\'answer_type\'], \'answer\': r[\'answer\'], \'prediction\': labels[p], \'correct\': p == t})\n        acc = {k.lower().replace(\'all\', \'overall\'): 100.0 * hit[k] / total[k] if total[k] else 0.0 for k in (\'OPEN\', \'CLOSED\', \'ALL\')}\n        return {\'open\': acc[\'open\'], \'closed\': acc[\'closed\'], \'overall\': acc[\'overall\']}, predictions\n\n    state_path, history = out / \'last.pt\', []\n    start = 0\n    if state_path.exists():  # only our own checkpoint from an earlier epoch of this run\n        state = torch.load(state_path, map_location=\'cpu\', weights_only=False)\n        model.load_state_dict(state[\'model\']); optimizer.load_state_dict(state[\'optimizer\'])\n        scheduler.load_state_dict(state[\'scheduler\']); scaler.load_state_dict(state[\'scaler\'])\n        start, history = state[\'epoch\'], state[\'history\']\n        torch.set_rng_state(state[\'rng\']); random.setstate(state[\'py_rng\']); np.random.set_state(state[\'np_rng\'])\n        if device == \'cuda\': torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n        print(\'resumed after epoch\', start)\n    for epoch in range(start, cfg[\'epochs\']):\n        model.train(); optimizer.zero_grad(set_to_none=True); loss_sum = seen = hits = 0\n        batches = len(loaders[\'train\'])\n        for step, (pixels, ids, mask, target, items) in enumerate(loaders[\'train\']):\n            with torch.autocast(device_type=\'cuda\', dtype=torch.float16, enabled=use_amp):\n                logits = model(pixels.to(device), ids.to(device), mask.to(device))\n            targets = F.one_hot(target.to(device), len(labels)).float()  # M3AE get_score() = 1.0\n            loss = F.binary_cross_entropy_with_logits(logits.float(), targets) * targets.shape[1]\n            window = min(accumulation, batches - (step // accumulation) * accumulation)\n            scaler.scale(loss / window).backward()\n            if (step + 1) % accumulation == 0 or step + 1 == batches:\n                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()\n            loss_sum += float(loss) * len(items); seen += len(items)\n            hits += int((logits.detach().argmax(-1).cpu() == target).sum())  # train mode (dropout on), like QCR Train_Acc\n        row = {\'epoch\': epoch, \'train_loss\': loss_sum / seen, \'train_acc\': 100.0 * hits / seen, \'lr\': scheduler.get_last_lr()[0]}\n        for split in (\'validation\', \'test\'):\n            if split in loaders:\n                row[split], predictions = evaluate(split)\n                if split == \'test\':\n                    (out / f\'test_predictions_epoch{epoch}.json\').write_text(json.dumps(predictions), encoding=\'utf-8\')\n        history.append(row)\n        print(json.dumps(row), flush=True)\n        state = {\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(),\n                 \'scaler\': scaler.state_dict(), \'epoch\': epoch + 1, \'history\': history, \'rng\': torch.get_rng_state(),\n                 \'py_rng\': random.getstate(), \'np_rng\': np.random.get_state(),\n                 \'cuda_rng\': torch.cuda.get_rng_state_all() if device == \'cuda\' else None}\n        torch.save(state, out / \'last.tmp\'); (out / \'last.tmp\').replace(state_path)\n        _write_json(out / \'history.json\', history)  # atomic: the monitor cell reads it while training runs\n    result = summarize_history(history)\n    result.update(dataset=dataset, run=run_index, config=cfg, full_protocol=full_protocol and len(history) == cfg[\'epochs\'],\n                  num_labels=len(labels), truncated_questions=truncated, counts={k: len(v) for k, v in splits.items()},\n                  paper_open_closed_overall=PAPER[dataset], output_dir=str(out),\n                  weights=list(BIOMEDCLIP[:3]), deviations=DEVIATIONS)\n    _write_json(out / \'result.json\', result)\n    return result\n\n\ndef config_for(dataset, overrides=None):\n    return {**DEFAULTS[\'common\'], **DEFAULTS[dataset], **(overrides or {})}\n\n\ndef run_dir(work_dir, dataset, run_index=0, overrides=None, limit=None):\n    """Output folder of one run: runs/biomedclip_meter_<dataset>_run<i>/full, or quick_<overrides> if anything changed."""\n    tag = \'full\' if not overrides and limit is None else \'quick_\' + \'_\'.join(f\'{k}{v}\' for k, v in sorted((overrides or {}).items()))\n    return Path(work_dir) / \'runs\' / f\'biomedclip_meter_{dataset}_run{run_index}\' / tag\n\n\ndef _write_json(path, obj):\n    path = Path(path)\n    tmp = path.with_name(path.name + \'.tmp\')\n    tmp.write_text(json.dumps(obj, indent=2), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\n# ---- background runner -----------------------------------------------------------------------------------\n# Training runs in a separate process (like the QCR reproduction, where main/main.py kept training after the\n# notebook cell lost its output), so the notebook stays usable for monitoring cells while it trains.\nDONE_MARK, START_MARK = \'ALL_JOBS_DONE\', \'=== trainer start\'\n\n\ndef job_key(job):\n    return job[\'dataset\'], job[\'run\'], json.dumps(job.get(\'overrides\') or {}, sort_keys=True)\n\n\ndef run_jobs(jobs, work_dir, cache_dir, micro_batch=None):\n    """Run jobs in order; each finished result is appended to work_dir/results.json and finished jobs are skipped."""\n    results_path = Path(work_dir) / \'results.json\'\n    for job in jobs:\n        results = json.loads(results_path.read_text(encoding=\'utf-8\')) if results_path.exists() else []\n        if any(job_key(r) == job_key(job) for r in results):\n            print(\'skip finished\', job_key(job), flush=True)\n            continue\n        print(\'\\n=== start\', job_key(job), time.strftime(\'%Y-%m-%d %H:%M:%S\'), \'===\', flush=True)\n        result = fit(job[\'dataset\'], work_dir, cache_dir, run_index=job[\'run\'], overrides=job.get(\'overrides\') or None,\n                     micro_batch=micro_batch)\n        result[\'overrides\'] = job.get(\'overrides\') or {}\n        results.append(result)\n        _write_json(results_path, results)\n        print(\'RESULT\', json.dumps({k: result[k] for k in (\'dataset\', \'run\', \'best_test_epoch\', \'last_epoch\', \'full_protocol\')}), flush=True)\n\n\ndef _pid_alive(pid):\n    """Linux /proc check; a finished child that was not reaped yet (zombie) counts as not running."""\n    try:\n        stat = Path(f\'/proc/{pid}/stat\').read_text()\n        cmdline = Path(f\'/proc/{pid}/cmdline\').read_bytes()\n    except OSError:\n        return False\n    return stat.rsplit(\')\', 1)[1].split()[0] != \'Z\' and b\'biomedclip_meter\' in cmdline\n\n\ndef trainer_pid(work_dir):\n    path = Path(work_dir) / \'train.pid\'\n    if path.exists():\n        pid = int(path.read_text().strip())\n        if _pid_alive(pid):\n            return pid\n    return None\n\n\ndef start_trainer(work_dir, cache_dir, jobs, micro_batch=None):\n    """Start run_jobs in a background process (log: work_dir/train.log). Does nothing if one is already running."""\n    work_dir = Path(work_dir)\n    pid = trainer_pid(work_dir)\n    if pid:\n        print(\'trainer already running, pid\', pid)\n        return pid\n    spec = work_dir / \'jobs.json\'\n    _write_json(spec, {\'work_dir\': str(work_dir), \'cache_dir\': str(cache_dir), \'micro_batch\': micro_batch, \'jobs\': jobs})\n    with open(work_dir / \'train.log\', \'a\', encoding=\'utf-8\') as log:\n        log.write(f\'\\n{START_MARK} {time.strftime("%Y-%m-%d %H:%M:%S")} ===\\n\')\n        log.flush()\n        proc = subprocess.Popen([sys.executable, \'-u\', str(Path(__file__).resolve()), str(spec)], cwd=str(Path(__file__).resolve().parent),\n                                stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True,\n                                env=dict(os.environ, PYTHONUNBUFFERED=\'1\'))\n    (work_dir / \'train.pid\').write_text(str(proc.pid))\n    print(\'trainer started, pid\', proc.pid, \'| log\', work_dir / \'train.log\')\n    return proc.pid\n\n\ndef stop_trainer(work_dir):\n    import signal\n    pid = trainer_pid(work_dir)\n    if pid:\n        os.killpg(pid, signal.SIGTERM)\n        print(\'sent SIGTERM to\', pid, \'(the next start resumes from the last finished epoch)\')\n    else:\n        print(\'no trainer running\')\n\n\ndef trainer_status(work_dir, tail_lines=15):\n    """running / finished (DONE_MARK after the latest start) / failed, plus the last log lines."""\n    log = Path(work_dir) / \'train.log\'\n    text = log.read_text(encoding=\'utf-8\', errors=\'replace\') if log.exists() else \'\'\n    latest = text.rsplit(START_MARK, 1)[-1] if START_MARK in text else \'\'\n    state = \'running\' if trainer_pid(work_dir) else \'finished\' if DONE_MARK in latest else \'failed\' if latest else \'not started\'\n    return {\'state\': state, \'tail\': \'\\n\'.join(latest.strip().splitlines()[-tail_lines:])}\n\n\ndef progress(work_dir, jobs):\n    """Per job: epochs done, best-so-far and latest test overall, minutes since the last epoch was written."""\n    rows = []\n    for job in jobs:\n        overrides = job.get(\'overrides\') or None\n        out = run_dir(work_dir, job[\'dataset\'], job[\'run\'], overrides)\n        hist_path = out / \'history.json\'\n        hist = json.loads(hist_path.read_text(encoding=\'utf-8\')) if hist_path.exists() else []\n        best = summarize_history(hist)[\'best_test_epoch\'] if hist else None\n        rows.append({\'dataset\': job[\'dataset\'], \'run\': job[\'run\'], \'overrides\': overrides or {},\n                     \'epochs\': f"{len(hist)}/{config_for(job[\'dataset\'], overrides)[\'epochs\']}",\n                     \'best_overall\': round(best[\'overall\'], 2) if best else None, \'best_epoch\': best[\'epoch\'] if best else None,\n                     \'last_overall\': round(hist[-1][\'test\'][\'overall\'], 2) if hist else None,\n                     \'last_train_acc\': round(hist[-1][\'train_acc\'], 2) if hist and \'train_acc\' in hist[-1] else None,\n                     \'min_since_update\': round((time.time() - hist_path.stat().st_mtime) / 60, 1) if hist else None})\n    return rows\n\n\n# ---- backup printed into the notebook output (read back with extract_notebook_backup.py) -------------------\nBACKUP_BEGIN, BACKUP_END = \'===QCR_BACKUP_BEGIN===\', \'===QCR_BACKUP_END===\'\n\n\ndef backup_text(work_dir, max_log_bytes=1 << 20):\n    """Small zip (results, histories, predictions of the best/last/val-selected epochs, log tail) as base64 text."""\n    work_dir = Path(work_dir)\n    buf = io.BytesIO()\n    with zipfile.ZipFile(buf, \'w\', compression=zipfile.ZIP_DEFLATED) as z:\n        for name in (\'results.json\', \'summary.json\', \'jobs.json\'):\n            if (work_dir / name).exists():\n                z.write(work_dir / name, name)\n        if (work_dir / \'train.log\').exists():\n            z.writestr(\'train.log\', (work_dir / \'train.log\').read_bytes()[-max_log_bytes:])\n        for hist_path in sorted(work_dir.glob(\'runs/*/*/history.json\')):\n            out = hist_path.parent\n            rel = out.relative_to(work_dir).as_posix()\n            hist = json.loads(hist_path.read_text(encoding=\'utf-8\'))\n            z.write(hist_path, f\'{rel}/history.json\')\n            if (out / \'result.json\').exists():\n                z.write(out / \'result.json\', f\'{rel}/result.json\')\n            if hist:\n                s = summarize_history(hist)\n                epochs = {s[\'best_test_epoch\'][\'epoch\'], hist[-1][\'epoch\']} | ({s[\'validation_selected\'][\'epoch\']} if \'validation_selected\' in s else set())\n                for e in sorted(epochs):\n                    p = out / f\'test_predictions_epoch{e}.json\'\n                    if p.exists():\n                        z.write(p, f\'{rel}/{p.name}\')\n    data = buf.getvalue()\n    b64 = base64.b64encode(data).decode(\'ascii\')\n    lines = [b64[i:i + 100] for i in range(0, len(b64), 100)]\n    return \'\\n\'.join([BACKUP_BEGIN, f\'biomedclip_backup.zip sha256={hashlib.sha256(data).hexdigest()}\', *lines, BACKUP_END])\n\n\ndef summarize_history(history):\n    best = None\n    for row in history:  # QCR/M3AE: first epoch with strictly higher test overall accuracy\n        if best is None or row[\'test\'][\'overall\'] > best[\'test\'][\'overall\']:\n            best = row\n    result = {\'epochs_logged\': len(history), \'best_test_epoch\': best and {\'epoch\': best[\'epoch\'], **best[\'test\']},\n              \'last_epoch\': history[-1][\'test\'] if history else None}\n    if history and \'validation\' in history[0]:\n        chosen = max(history, key=lambda r: (r[\'validation\'][\'overall\'], -r[\'epoch\']))\n        result[\'validation_selected\'] = {\'epoch\': chosen[\'epoch\'], **chosen[\'test\']}\n    return result\n\n\ndef summarize(results):\n    import statistics\n    table = {}\n    for dataset in PAPER:\n        rows = [r for r in results if r[\'dataset\'] == dataset]\n        if not rows:\n            continue\n        entry = {\'runs\': len(rows), \'full_protocol\': all(r[\'full_protocol\'] for r in rows), \'paper_open_closed_overall\': PAPER[dataset]}\n        for protocol in (\'best_test_epoch\', \'last_epoch\', \'validation_selected\'):\n            if all(protocol in r for r in rows):\n                entry[protocol] = {k: statistics.mean(r[protocol][k] for r in rows) for k in (\'open\', \'closed\', \'overall\')}\n                if len(rows) > 1:\n                    entry[protocol][\'overall_std\'] = statistics.stdev(r[protocol][\'overall\'] for r in rows)\n        table[dataset] = entry\n    return table\n\n\nDEVIATIONS = [\n    \'Public 224px BiomedCLIP checkpoint; the paper used an unreleased ViT-B/16-448 variant\',\n    \'METER fusion re-implemented and randomly initialised; the paper gives no fusion details beyond METER\',\n    \'Fine-tuning hyper-parameters, answer set and preprocessing taken from M3AE (METER-based medical VQA); the paper states none\',\n    \'Model selection: best test epoch as in QCR/M3AE (paper does not say); last epoch and SLAKE validation-selected epoch also reported\',\n    \'Open/Closed split by the ground-truth answer_type (VQA-RAD test 179/272); the PubMedCLIP QCR code splits by its \'\n    \'question-type classifier prediction (logged 180/271), so only Overall shares the same denominator\',\n]\n\n\nif __name__ == \'__main__\':  # background entry point used by start_trainer\n    spec = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\n    run_jobs(spec[\'jobs\'], spec[\'work_dir\'], spec[\'cache_dir\'], spec.get(\'micro_batch\'))\n    print(DONE_MARK, time.strftime(\'%Y-%m-%d %H:%M:%S\'), flush=True)\n\n', encoding='utf-8')
(CODE_ROOT / 'biomed_vital_qlora.py').write_text('"""BioMed-VITAL (13B) downstream VQA fine-tuning with QLoRA: an APPROXIMATION of the paper row.\n\nWhat is public: only the instruction-tuned checkpoint (HF mao1207/BioMed-VITAL-models, LLaVA-Med v1\narchitecture: LLaMA-13B + one Linear mm_projector + CLIP ViT-L/14-336, <im_patch> tokens) and the authors\'\ninference script (eval/model_vqa_med.py). No VQA fine-tuned weights, no fine-tuning code. The paper says\n"fine-tuned for 15 epochs with a learning rate of 2e-5" and "Following the LLaVA-Med approach".\n\nThis module therefore follows the LLaVA-Med v1.0.0 code for everything the paper leaves out:\n  * training text / label masking: llava/train/train.py preprocess() (v0 path, "### Human: ... ### Assistant: ...");\n  * image tokens: <im_start> + 576 x <im_patch> + <im_end>, patch embeddings replaced by projected CLIP\n    features (hidden_states[-2], CLS dropped), as LlavaLlamaModel.forward;\n  * inference prompt and decoding: BioMed-VITAL eval/model_vqa_med.py (conv "default" = conv_v1_2,\n    do_sample=True, temperature 0.7, max_new_tokens 1024, stop at "###");\n  * paper metrics: LLaVA-Med llava/eval/run_eval.py (closed: yes/no hit, open: token recall);\nand replaces full fine-tuning by QLoRA (LLaVA v1.5 task-LoRA recipe: r 128, alpha 256, dropout 0.05,\nall LLM linear layers, projector trained at 2e-5; 4-bit NF4 double-quant, bf16 compute).\n\nProtocol (agreed with the professor): LoRA lr in {2e-5, 1e-4, 2e-4}, everything else identical. Every\nrun uses a 3-epoch cosine schedule (LLaVA-Med v1 command: num_train_epochs 3). Each lr is trained for\n1 epoch on VQA-RAD train minus a held-out validation split; the lr with the best validation score is\ncontinued (same optimizer/scheduler state) to epoch 3, so that run is identical to a 3-epoch run.\nSLAKE is then trained 3 epochs with the chosen lr. Test sets are evaluated once, at the end.\nVQA-RAD = the original release split (1,797 / 451; MEVF \'frame\' questions removed), as in the paper.\n"""\nimport base64\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport random\nimport re\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path\n\nimport qcr_port\nimport biomedclip_meter  # normalize_answer (VQA/M3AE/LLaVA-Med normalisation) and the SLAKE validation asset\n\nMODEL = (\'mao1207/BioMed-VITAL-models\', \'c93f2607fdf20b16f3874e8fd5183ddffe705a9c\')\nMODEL_PATTERNS = [\'config.json\', \'generation_config.json\', \'tokenizer.model\', \'tokenizer_config.json\',\n                  \'special_tokens_map.json\', \'added_tokens.json\', \'pytorch_model.bin.index.json\', \'pytorch_model-*.bin\']\nVISION = (\'openai/clip-vit-large-patch14-336\', \'ce19dc912ca5cd21c8a653c79e251e808ccabcd1\')\n\n# BioMed-VITAL paper Table 4 (13B, 60K) = Expert-CFG Table 1: [open, closed, overall].\nPAPER = {\'rad\': [64.88, 84.55, 76.74], \'slake\': [87.82, 86.54, 87.32]}\nSOURCE_COUNTS = {\'rad\': {\'train\': 1797, \'test\': 451}, \'slake\': {\'train\': 4919, \'validation\': 1053, \'test\': 1061}}\n\nDEFAULTS = dict(\n    epochs=3, select_epochs=1, lrs=[2e-5, 1e-4, 2e-4], projector_lr=2e-5,\n    global_batch=64, micro_batch=4, weight_decay=0.0, warmup_ratio=0.03, max_grad_norm=1.0, max_length=2048,\n    lora_r=128, lora_alpha=256, lora_dropout=0.05, seed=0, rad_val_fraction=0.1,\n    eval_batch=8, val_max_new_tokens=128, paper_max_new_tokens=1024, temperature=0.7, save_every_steps=10,\n)\n\n# ---- LLaVA-Med v1.0.0 text format -------------------------------------------------------------------------\nIGNORE_INDEX = -100\nIMAGE_TOKEN, IM_PATCH, IM_START, IM_END = \'<image>\', \'<im_patch>\', \'<im_start>\', \'<im_end>\'\nSEP = \'###\'\n# llava/conversation.py conv_v1_2 (= conv_templates["default"], used by BioMed-VITAL eval/model_vqa_med.py)\nSYSTEM = ("A chat between a curious human and an artificial intelligence assistant. "\n          "The assistant gives helpful, detailed, and polite answers to the human\'s questions.")\nFEWSHOT = (\n    (\'Human\', \'What are the key differences between renewable and non-renewable energy sources?\'),\n    (\'Assistant\',\n     \'Renewable energy sources are those that can be replenished naturally in a relatively \'\n     \'short amount of time, such as solar, wind, hydro, geothermal, and biomass. \'\n     \'Non-renewable energy sources, on the other hand, are finite and will eventually be \'\n     \'depleted, such as coal, oil, and natural gas. Here are some key differences between \'\n     \'renewable and non-renewable energy sources:\\n\'\n     \'1. Availability: Renewable energy sources are virtually inexhaustible, while non-renewable \'\n     \'energy sources are finite and will eventually run out.\\n\'\n     \'2. Environmental impact: Renewable energy sources have a much lower environmental impact \'\n     \'than non-renewable sources, which can lead to air and water pollution, greenhouse gas emissions, \'\n     \'and other negative effects.\\n\'\n     \'3. Cost: Renewable energy sources can be more expensive to initially set up, but they typically \'\n     \'have lower operational costs than non-renewable sources.\\n\'\n     \'4. Reliability: Renewable energy sources are often more reliable and can be used in more remote \'\n     \'locations than non-renewable sources.\\n\'\n     \'5. Flexibility: Renewable energy sources are often more flexible and can be adapted to different \'\n     \'situations and needs, while non-renewable sources are more rigid and inflexible.\\n\'\n     \'6. Sustainability: Renewable energy sources are more sustainable over the long term, while \'\n     \'non-renewable sources are not, and their depletion can lead to economic and social instability.\\n\'),\n)\n\n\ndef image_placeholder(n_patches):\n    return IM_START + IM_PATCH * n_patches + IM_END  # mm_use_im_start_end = true in the checkpoint config\n\n\ndef _n_tokens(tokenizer, text, max_length):\n    """train.py _tokenize_fn: length = non-pad tokens of tokenizer(text) (BOS included, pad = <unk>)."""\n    ids = tokenizer(text, max_length=max_length, truncation=True).input_ids\n    return sum(1 for t in ids if t != tokenizer.pad_token_id)\n\n\ndef build_train_example(tokenizer, question, answer, n_patches, max_length):\n    """LLaVA-Med v1 train.py: preprocess_multimodal + preprocess (v0: _add_speaker_and_signal, _mask_targets).\n\n    The human turn is "<question>\\\\n<image>" so the image follows the question exactly as at inference\n    (model_vqa_med.py appends the image tokens after the question).\n    """\n    human = f\'{question}\\n{IMAGE_TOKEN}\'.replace(IMAGE_TOKEN, image_placeholder(n_patches))\n    header = f\'{SYSTEM}\\n\\n\'\n    sentences = [(\'human\', f\'### Human: {human}\\n\'), (\'gpt\', f\'### Assistant: {answer}\\n\')]\n    conversation = header + \'\'.join(s for _, s in sentences) + \'### \'\n    input_ids = tokenizer(conversation, max_length=max_length, truncation=True).input_ids\n    labels = list(input_ids)\n    lens = [_n_tokens(tokenizer, header, max_length)] + [_n_tokens(tokenizer, s, max_length) for _, s in sentences]\n    cur = lens[0]\n    labels[:cur] = [IGNORE_INDEX] * min(cur, len(labels))\n    for length, (speaker, _) in zip(lens[1:], sentences):\n        if speaker == \'human\':\n            for i in range(cur + 2, min(cur + length, len(labels))):\n                labels[i] = IGNORE_INDEX\n        cur += length\n    return input_ids, labels\n\n\ndef build_eval_prompt(question, n_patches):\n    """BioMed-VITAL eval/model_vqa_med.py with conv_mode \'default\' (conv_v1_2, SeparatorStyle.SINGLE)."""\n    qs = question.replace(IMAGE_TOKEN, \'\').strip() + \'\\n\' + image_placeholder(n_patches)\n    ret = SYSTEM + SEP\n    for role, message in FEWSHOT + ((\'Human\', qs),):\n        ret += role + \': \' + message + SEP\n    return ret\n\n\ndef cut_output(text):\n    """model_vqa_med.py: keep the text before the first conv.sep (\'###\'), stripped.\n\n    The fine-tuned model was trained on "...<im_end>\\\\n### Assistant: a" but the eval prompt already ends in \'###\',\n    so it may answer "### Assistant: a"; a leading separator is dropped first (as the simple-mode loop of\n    model_vqa_med.py does), otherwise the answer would be cut to \'\'.\n    """\n    text = text.strip()\n    while text.startswith(SEP):\n        text = text[len(SEP):].strip()\n    index = text.find(SEP)\n    return (text if index < 0 else text[:index]).strip()\n\n\n# ---- metrics -------------------------------------------------------------------------------------------------\nnormalize_word = biomedclip_meter.normalize_answer  # same tables as LLaVA-Med eval_metrics/glossary.py\n\n\ndef _split_counts(sentence):\n    return Counter(sentence.lower().strip().split())  # eval_metrics/utils.py split_sentence(sentence, 1)\n\n\ndef llava_f1(candidate, reference):\n    """LLaVA-Med eval_metrics calculate_f1score -> (f1, precision, recall), counting as the original."""\n    candidate_words = _split_counts(normalize_word(candidate))\n    reference_words = _split_counts(normalize_word(reference))\n    tp = fp = fn = 0\n    for word in set(candidate_words) | set(reference_words):\n        if word in candidate_words and word in reference_words:\n            tp += candidate_words[word]\n        elif word in candidate_words:\n            fp += candidate_words[word]\n        else:\n            fn += reference_words[word]\n    if not candidate_words or not reference_words or tp == 0:\n        return 0.0, 0.0, 0.0\n    precision, recall = tp / (tp + fp), tp / (tp + fn)\n    return 2 * precision * recall / (precision + recall), precision, recall\n\n\ndef paper_score(answer_type, prediction, answer):\n    """run_eval.py: closed = yes/no hit, open = token recall (the paper\'s \'recall\' for open questions)."""\n    gt, pred = normalize_word(str(answer).lower()), normalize_word(str(prediction).lower())\n    if answer_type == \'CLOSED\':\n        return float((\'yes\' in pred or \'no\' in pred) and gt in pred)\n    return llava_f1(pred, gt)[2]\n\n\n_ROLE_PREFIX = re.compile(r\'^\\s*(assistant|gpt)\\s*:\\s*\', re.I)\n\n\ndef extract_answer(prediction):\n    """For the unified metrics only: drop a leading role tag the model may emit (\'Assistant: ...\')."""\n    return _ROLE_PREFIX.sub(\'\', prediction).strip()\n\n\ndef token_f1(prediction, answer):\n    p, g = normalize_word(prediction).split(), normalize_word(answer).split()\n    common = sum((Counter(p) & Counter(g)).values())\n    if not p or not g or common == 0:\n        return 0.0\n    precision, recall = common / len(p), common / len(g)\n    return 2 * precision * recall / (precision + recall)\n\n\ndef score(rows, predictions):\n    """Paper protocol (run_eval.py) and the unified protocol (same normalisation for every model).\n\n    Overall = mean over all questions, i.e. Open and Closed weighted by their counts, as in the paper tables.\n    Open/Closed always follow the ground-truth answer_type.\n    """\n    by_type = {\'OPEN\': [], \'CLOSED\': []}\n    per_question = []\n    for r in rows:\n        pred = predictions[str(r[\'qid\'])]\n        clean = extract_answer(pred)\n        em = float(normalize_word(clean) == normalize_word(r[\'answer\']))\n        item = {\'qid\': r[\'qid\'], \'answer_type\': r[\'answer_type\'], \'answer\': r[\'answer\'], \'prediction\': pred,\n                \'paper\': paper_score(r[\'answer_type\'], pred, r[\'answer\']), \'em\': em, \'f1\': token_f1(clean, r[\'answer\'])}\n        by_type[r[\'answer_type\']].append(item)\n        per_question.append(item)\n    mean = lambda xs: 100.0 * sum(xs) / len(xs) if xs else 0.0\n    allq = by_type[\'OPEN\'] + by_type[\'CLOSED\']\n    result = {\n        \'counts\': {\'open\': len(by_type[\'OPEN\']), \'closed\': len(by_type[\'CLOSED\'])},\n        \'paper\': {\'open\': mean([i[\'paper\'] for i in by_type[\'OPEN\']]), \'closed\': mean([i[\'paper\'] for i in by_type[\'CLOSED\']]),\n                  \'overall\': mean([i[\'paper\'] for i in allq])},\n        \'unified\': {\'closed_acc\': mean([i[\'em\'] for i in by_type[\'CLOSED\']]), \'open_em\': mean([i[\'em\'] for i in by_type[\'OPEN\']]),\n                    \'open_f1\': mean([i[\'f1\'] for i in by_type[\'OPEN\']]), \'overall_em\': mean([i[\'em\'] for i in allq])},\n    }\n    return result, per_question\n\n\ndef selection_key(val_result):\n    """Validation rule for choosing the lr: unified overall EM, then open token F1."""\n    u = val_result[\'unified\']\n    return (round(u[\'overall_em\'], 6), round(u[\'open_f1\'], 6))\n\n\n# ---- data ----------------------------------------------------------------------------------------------------\ndef _rows(records, image_root, image_key):\n    rows = []\n    for r in records:\n        kind = str(r[\'answer_type\']).strip().upper()\n        if kind not in (\'OPEN\', \'CLOSED\'):\n            raise ValueError(f\'Bad answer_type {r["answer_type"]!r}\')\n        rows.append(dict(qid=r[\'qid\'], question=r[\'question\'], answer=str(r[\'answer\']), answer_type=kind,\n                         image_name=r[image_key], image=str(Path(image_root) / r[image_key]), phrase_type=r.get(\'phrase_type\')))\n    return rows\n\n\ndef split_validation(rows, fraction, seed):\n    """Hold out whole images (no image shared between train and validation)."""\n    images = sorted({r[\'image_name\'] for r in rows})\n    random.Random(seed).shuffle(images)\n    held = set(images[:max(1, round(len(images) * fraction))])\n    return [r for r in rows if r[\'image_name\'] not in held], [r for r in rows if r[\'image_name\'] in held]\n\n\ndef load_data(dataset, work_dir, cache_dir, cfg=None, rad_dir=None):\n    """VQA-RAD: original split (MEVF/QCR files with phrase_type \'frame\' removed) + image-level validation.\n    SLAKE: BoKelvin/SLAKE English train/validation/test. Files pinned and hashed (qcr_port)."""\n    cfg = {**DEFAULTS, **(cfg or {})}\n    work_dir, cache_dir = Path(work_dir), Path(cache_dir)\n    if dataset == \'rad\':\n        data = Path(rad_dir) if rad_dir else qcr_port._clone_original_qcr(work_dir) / \'data\'\n        for name in (\'trainset.json\', \'testset.json\'):\n            if qcr_port.sha256(data / name) != qcr_port.RAD_SHA256[name]:\n                raise ValueError(\'VQA-RAD split file changed: \' + name)\n        train = [r for r in json.loads((data / \'trainset.json\').read_text(encoding=\'utf-8\')) if r[\'phrase_type\'] != \'frame\']\n        test = json.loads((data / \'testset.json\').read_text(encoding=\'utf-8\'))\n        source = {\'train\': len(train), \'test\': len(test)}\n        train, test = _rows(train, data / \'images\', \'image_name\'), _rows(test, data / \'images\', \'image_name\')\n        train, validation = split_validation(train, cfg[\'rad_val_fraction\'], cfg[\'seed\'])\n        splits = {\'train\': train, \'validation\': validation, \'test\': test}\n    elif dataset == \'slake\':\n        root = work_dir / \'slake\'\n        if not (root / \'imgs\').exists():\n            root.mkdir(parents=True, exist_ok=True)\n            qcr_port.safe_extract(qcr_port.fetch(\'slake_imgs\', cache_dir), root)\n        splits = {}\n        for split, key in ((\'train\', \'slake_train\'), (\'validation\', \'slake_validation\'), (\'test\', \'slake_test\')):\n            records = [r for r in json.loads(Path(qcr_port.fetch(key, cache_dir)).read_text(encoding=\'utf-8\')) if r[\'q_lang\'] == \'en\']\n            splits[split] = _rows(records, root / \'imgs\', \'img_name\')\n        source = {k: len(v) for k, v in splits.items()}\n    else:\n        raise ValueError(\'dataset must be rad or slake\')\n    if source != SOURCE_COUNTS[dataset]:\n        raise ValueError(f\'{dataset}: expected {SOURCE_COUNTS[dataset]}, got {source}\')\n    if dataset == \'rad\' and any(r[\'phrase_type\'] == \'frame\' for r in splits[\'test\']):\n        raise ValueError(\'VQA-RAD test unexpectedly contains frame questions\')\n    for rows in splits.values():\n        for r in rows:\n            if not Path(r[\'image\']).exists():\n                raise FileNotFoundError(r[\'image\'])\n    return splits\n\n\ndef data_summary(splits):\n    return {k: {\'questions\': len(v), \'images\': len({r[\'image_name\'] for r in v}),\n                \'open\': sum(r[\'answer_type\'] == \'OPEN\' for r in v), \'closed\': sum(r[\'answer_type\'] == \'CLOSED\' for r in v)}\n            for k, v in splits.items()}\n\n\n# ---- model ---------------------------------------------------------------------------------------------------\nLLAMA_KEYS = (\'vocab_size\', \'hidden_size\', \'intermediate_size\', \'num_hidden_layers\', \'num_attention_heads\', \'num_key_value_heads\',\n              \'hidden_act\', \'max_position_embeddings\', \'rms_norm_eps\', \'initializer_range\', \'bos_token_id\', \'eos_token_id\',\n              \'pad_token_id\', \'tie_word_embeddings\', \'rope_scaling\', \'pretraining_tp\')\nLORA_TARGETS = [\'q_proj\', \'k_proj\', \'v_proj\', \'o_proj\', \'gate_proj\', \'up_proj\', \'down_proj\']\n\n\ndef find_all_linear_names(model):\n    """LLaVA train.py find_all_linear_names (LLM linear layers, without lm_head / multimodal parts)."""\n    import torch\n    names = set()\n    for name, module in model.named_modules():\n        if any(k in name for k in (\'mm_projector\', \'vision_tower\', \'vision_resampler\')):\n            continue\n        if isinstance(module, torch.nn.Linear):\n            names.add(name.split(\'.\')[-1])\n    names.discard(\'lm_head\')\n    return sorted(names)\n\n\ndef download_model(cache_dir):\n    from huggingface_hub import snapshot_download\n    return Path(snapshot_download(MODEL[0], revision=MODEL[1], allow_patterns=MODEL_PATTERNS, cache_dir=str(cache_dir)))\n\n\nclass Components:\n    """tokenizer, image processor, frozen CLIP tower, trainable projector, LLM with LoRA."""\n\n    def __init__(self, tokenizer, image_processor, vision, projector, model, n_patches, device, initial_projector):\n        self.tokenizer, self.image_processor, self.vision, self.projector, self.model = tokenizer, image_processor, vision, projector, model\n        self.n_patches, self.device, self.initial_projector = n_patches, device, initial_projector\n        self.patch_id = tokenizer.convert_tokens_to_ids(IM_PATCH)\n        self.start_id, self.end_id = tokenizer.convert_tokens_to_ids([IM_START, IM_END])\n\n\ndef build_components(cfg, cache_dir, device=\'cuda\'):\n    """4-bit NF4 LLaMA-13B (LLaVA train.py BitsAndBytesConfig) + LoRA, fp32 projector, fp16 CLIP tower."""\n    import torch\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoTokenizer, BitsAndBytesConfig, CLIPImageProcessor, CLIPVisionModel, LlamaConfig, LlamaForCausalLM\n\n    model_dir = download_model(cache_dir)\n    raw = json.loads((model_dir / \'config.json\').read_text(encoding=\'utf-8\'))\n    assert raw[\'mm_use_im_start_end\'] and raw[\'mm_vision_select_layer\'] == -2 and raw[\'mm_vision_tower\'] == VISION[0]\n    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=False)\n    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_use_double_quant=True,\n                             bnb_4bit_compute_dtype=torch.bfloat16, llm_int8_skip_modules=[\'lm_head\'])\n    llm = LlamaForCausalLM.from_pretrained(model_dir, config=LlamaConfig(**{k: raw.get(k) for k in LLAMA_KEYS}),\n                                           quantization_config=bnb, torch_dtype=torch.bfloat16, device_map={\'\': 0},\n                                           low_cpu_mem_usage=True, attn_implementation=\'sdpa\')\n    llm = prepare_model_for_kbit_training(llm, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={\'use_reentrant\': False})\n    index = json.loads((model_dir / \'pytorch_model.bin.index.json\').read_text(encoding=\'utf-8\'))[\'weight_map\']\n    shard = torch.load(model_dir / index[\'model.mm_projector.weight\'], map_location=\'cpu\', weights_only=True, mmap=True)\n    projector_state = {\'weight\': shard[\'model.mm_projector.weight\'].float().clone(), \'bias\': shard[\'model.mm_projector.bias\'].float().clone()}\n    del shard\n    vision = CLIPVisionModel.from_pretrained(VISION[0], revision=VISION[1], torch_dtype=torch.float16, cache_dir=str(cache_dir))\n    image_processor = CLIPImageProcessor.from_pretrained(VISION[0], revision=VISION[1], cache_dir=str(cache_dir))\n    return _assemble(cfg, tokenizer, image_processor, vision, projector_state, llm, device)\n\n\ndef _assemble(cfg, tokenizer, image_processor, vision, projector_state, llm, device):\n    import torch\n    from peft import LoraConfig, get_peft_model\n    vision = vision.to(device).eval().requires_grad_(False)\n    n_patches = (vision.config.image_size // vision.config.patch_size) ** 2\n    projector = torch.nn.Linear(vision.config.hidden_size, llm.config.hidden_size).to(device)\n    projector.load_state_dict(projector_state)\n    targets = find_all_linear_names(llm)\n    if targets != sorted(LORA_TARGETS):\n        raise ValueError(f\'unexpected LoRA targets {targets}\')\n    lora = LoraConfig(r=cfg[\'lora_r\'], lora_alpha=cfg[\'lora_alpha\'], target_modules=targets, lora_dropout=cfg[\'lora_dropout\'],\n                      bias=\'none\', task_type=\'CAUSAL_LM\')\n    model = get_peft_model(llm, lora)\n    for tok in (IM_PATCH, IM_START, IM_END):\n        if tokenizer.convert_tokens_to_ids(tok) in (None, tokenizer.unk_token_id):\n            raise ValueError(\'tokenizer misses \' + tok)\n    return Components(tokenizer, image_processor, vision, projector, model, n_patches, device,\n                      {k: v.detach().cpu().clone() for k, v in projector.state_dict().items()})\n\n\ndef reset_trainable(c, seed):\n    """Fresh run: LoRA re-initialised (A kaiming, B zero, as peft does) and the checkpoint\'s projector."""\n    import torch\n    from peft.tuners.lora import LoraLayer\n    torch.manual_seed(seed)\n    for module in c.model.modules():\n        if isinstance(module, LoraLayer):\n            module.reset_lora_parameters(\'default\', True)\n    c.projector.load_state_dict(c.initial_projector)\n\n\ndef trainable_named(c):\n    params = [(f\'lora.{n}\', p) for n, p in c.model.named_parameters() if p.requires_grad]\n    params += [(f\'projector.{n}\', p) for n, p in c.projector.named_parameters()]\n    return params\n\n\ndef pixel_cache(c, rows):\n    import torch\n    from PIL import Image\n    cache = {}\n    for r in rows:\n        if r[\'image_name\'] not in cache:\n            with Image.open(r[\'image\']) as im:\n                cache[r[\'image_name\']] = c.image_processor.preprocess(im.convert(\'RGB\'), return_tensors=\'pt\')[\'pixel_values\'][0].to(torch.float16)\n    return cache\n\n\ndef embed(c, input_ids, pixel_values):\n    """LlavaLlamaModel.forward: word embeddings with the <im_patch> positions replaced by projected CLIP patch features."""\n    import torch\n    embeds = c.model.get_input_embeddings()(input_ids)\n    with torch.no_grad():\n        feats = c.vision(pixel_values.to(c.device, torch.float16), output_hidden_states=True).hidden_states[-2][:, 1:]\n    image = c.projector(feats.float())\n    mask = input_ids == c.patch_id\n    if not bool((mask.sum(1) == c.n_patches).all()):\n        raise ValueError(\'every sample needs exactly one image (n_patches <im_patch> tokens)\')\n    embeds = embeds.clone()\n    embeds[mask] = image.reshape(-1, image.shape[-1]).to(embeds.dtype)\n    return embeds\n\n\ndef _autocast(c):\n    import torch\n    return torch.autocast(device_type=\'cuda\', dtype=torch.bfloat16, enabled=c.device == \'cuda\')\n\n\n# ---- generation ------------------------------------------------------------------------------------------------\ndef generate(c, rows, pixels, mode, cfg, seed=0, log=print):\n    """Batched (left-padded) generation with the eval prompt. mode \'paper\': do_sample, T=0.7, 1024 new tokens;\n    mode \'greedy\': deterministic, used for validation (and as an extra test decoding)."""\n    import torch\n    from transformers import StoppingCriteria, StoppingCriteriaList\n\n    tok = c.tokenizer\n\n    class StopOnSep(StoppingCriteria):\n        def __call__(self, input_ids, scores, **kwargs):\n            return torch.tensor([SEP in t for t in tok.batch_decode(input_ids, skip_special_tokens=True)], device=input_ids.device)\n\n    kwargs = dict(do_sample=True, temperature=cfg[\'temperature\'], max_new_tokens=cfg[\'paper_max_new_tokens\']) if mode == \'paper\' \\\n        else dict(do_sample=False, max_new_tokens=cfg[\'val_max_new_tokens\'])\n    c.model.eval()\n    torch.manual_seed(seed)\n    out, raw, start = {}, {}, time.time()\n    for b in range(0, len(rows), cfg[\'eval_batch\']):\n        batch = rows[b:b + cfg[\'eval_batch\']]\n        prompts = [tok(build_eval_prompt(r[\'question\'], c.n_patches)).input_ids for r in batch]\n        width = max(map(len, prompts))\n        ids = torch.tensor([[tok.pad_token_id] * (width - len(p)) + p for p in prompts], device=c.device)\n        mask = torch.tensor([[0] * (width - len(p)) + [1] * len(p) for p in prompts], device=c.device)\n        pix = torch.stack([pixels[r[\'image_name\']] for r in batch])\n        with torch.inference_mode(), _autocast(c):\n            gen = c.model.generate(inputs_embeds=embed(c, ids, pix), attention_mask=mask, pad_token_id=tok.pad_token_id,\n                                   eos_token_id=tok.eos_token_id, use_cache=True,\n                                   stopping_criteria=StoppingCriteriaList([StopOnSep()]), **kwargs)\n        for r, text in zip(batch, tok.batch_decode(gen, skip_special_tokens=True)):\n            out[str(r[\'qid\'])] = cut_output(text)\n            raw[str(r[\'qid\'])] = text\n        if b == 0 or (b // cfg[\'eval_batch\']) % 10 == 0:\n            done = min(b + cfg[\'eval_batch\'], len(rows))\n            log(f\'  generate[{mode}] {done}/{len(rows)}  {(time.time() - start) / done:.2f}s/question  \'\n                f\'eta {(time.time() - start) / done * (len(rows) - done) / 60:.1f} min\')\n    return out, raw\n\n\n# ---- training --------------------------------------------------------------------------------------------------\ndef _collate(c, examples, pixels):\n    import torch\n    width = max(len(e[\'input_ids\']) for e in examples)\n    pad = c.tokenizer.pad_token_id\n    ids = torch.tensor([e[\'input_ids\'] + [pad] * (width - len(e[\'input_ids\'])) for e in examples], device=c.device)\n    labels = torch.tensor([e[\'labels\'] + [IGNORE_INDEX] * (width - len(e[\'labels\'])) for e in examples], device=c.device)\n    mask = torch.tensor([[1] * len(e[\'input_ids\']) + [0] * (width - len(e[\'input_ids\'])) for e in examples], device=c.device)\n    return ids, labels, mask, torch.stack([pixels[e[\'image_name\']] for e in examples])\n\n\ndef _write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    tmp.write_text(json.dumps(obj, indent=2), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\n# ---- saved models and Hugging Face Hub copy -------------------------------------------------------------------\n# The Colab disk disappears with the runtime and a Colab kernel inside VS Code cannot download files to the PC, so\n# everything worth keeping is also pushed to a private HF model repo (token: HF_TOKEN env) and pulled to the PC there.\nMODEL_FILES = (\'adapter_model.safetensors\', \'adapter_config.json\', \'mm_projector.safetensors\', \'meta.json\')\n\n\ndef export_model(c, out_dir, meta):\n    """Fine-tuned model = LoRA adapter (peft format) + projector; the 4-bit base comes from MODEL at load time."""\n    from safetensors.torch import save_file\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    c.model.save_pretrained(str(out))\n    save_file({k: v.detach().cpu().contiguous() for k, v in c.projector.state_dict().items()}, str(out / \'mm_projector.safetensors\'))\n    _write_json(out / \'meta.json\', {\'base_model\': MODEL[0], \'base_revision\': MODEL[1], \'vision_tower\': VISION[0],\n                                    \'vision_revision\': VISION[1], **meta})\n\n\ndef load_model(c, model_dir):\n    """Put a model saved by export_model into the components (for re-evaluation without retraining)."""\n    import torch\n    from peft import set_peft_model_state_dict\n    from safetensors.torch import load_file\n    d = Path(model_dir)\n    result = set_peft_model_state_dict(c.model, load_file(str(d / \'adapter_model.safetensors\')))\n    if getattr(result, \'unexpected_keys\', None):\n        raise ValueError(f\'adapter keys do not match the model: {result.unexpected_keys[:5]}\')\n    c.projector.load_state_dict({k: v.to(c.device, torch.float32) for k, v in load_file(str(d / \'mm_projector.safetensors\')).items()})\n    return json.loads((d / \'meta.json\').read_text(encoding=\'utf-8\'))\n\n\nclass Hub:\n    """Copies work_dir files to <repo>/<prefix>/ (same relative paths) and restores missing ones on a new runtime."""\n\n    def __init__(self, repo_id, prefix=\'\', upload_state=True, log=print):\n        from huggingface_hub import HfApi\n        self.api, self.repo, self.prefix, self.upload_state, self.log = HfApi(), repo_id, prefix.strip(\'/\'), upload_state, log\n\n    def remote(self, rel):\n        return f\'{self.prefix}/{rel}\' if self.prefix else rel\n\n    def upload(self, work, rels, message):\n        """One commit; failures are logged, never stop training (hub_sync retries everything later)."""\n        from huggingface_hub import CommitOperationAdd\n        work = Path(work)\n        rels = [r for r in dict.fromkeys(rels) if (work / r).is_file() and (self.upload_state or not r.endswith(\'state.pt\'))]\n        if not rels:\n            return True\n        for attempt in range(1, 4):\n            try:\n                t0 = time.time()\n                ops = [CommitOperationAdd(self.remote(r), str(work / r)) for r in rels]\n                self.api.create_commit(self.repo, ops, commit_message=message, repo_type=\'model\')\n                mb = sum((work / r).stat().st_size for r in rels) / 1e6\n                self.log(f\'hub: {message} ({len(rels)} files, {mb:.0f} MB, {time.time() - t0:.0f}s)\')\n                return True\n            except Exception as e:  # network / rate limit: retry, then leave it to hub_sync\n                self.log(f\'hub upload failed ({attempt}/3, {message}): {e!r}\')\n                time.sleep(30 * attempt)\n        return False\n\n    def restore(self, work):\n        """Download results, run progress and resume states that this runtime does not have (new Colab runtime)."""\n        from huggingface_hub import hf_hub_download\n        work = Path(work)\n        files = self.api.list_repo_files(self.repo, repo_type=\'model\')\n        head = f\'{self.prefix}/\' if self.prefix else \'\'\n        wanted = [f[len(head):] for f in files if f.startswith(head)]\n        wanted = [r for r in wanted if re.fullmatch(r\'results/[^/]+\\.json|runs/[^/]+/(progress\\.json|state\\.pt)\', r)\n                  and not (work / r).exists()]\n        tmp = work / \'.hub_tmp\'\n        for r in wanted:\n            path = Path(hf_hub_download(self.repo, self.remote(r), repo_type=\'model\', local_dir=str(tmp)))\n            (work / r).parent.mkdir(parents=True, exist_ok=True)\n            path.replace(work / r)\n            self.log(f\'hub: restored {r}\')\n        if tmp.exists():\n            import shutil\n            shutil.rmtree(tmp, ignore_errors=True)\n        return wanted\n\n\ndef hub_files(work_dir, include_state=True):\n    """Every file hub_sync keeps on the Hub (relative to work_dir)."""\n    work = Path(work_dir)\n    pats = [\'summary.json\', \'spec.json\', \'train.log\', \'results/*.json\', \'runs/*/progress.json\', \'runs/*/epoch*/*\']\n    pats += [\'runs/*/state.pt\'] if include_state else []\n    return sorted({p.relative_to(work).as_posix() for pat in pats for p in work.glob(pat) if p.is_file()})\n\n\ndef make_hub(spec, log=print):\n    if not spec.get(\'hub_repo\'):\n        return None\n    return Hub(spec[\'hub_repo\'], spec.get(\'hub_prefix\', Path(spec[\'work_dir\']).name), spec.get(\'hub_state\', True), log=log)\n\n\ndef hub_sync(spec, log=print):\n    """Upload everything (again); files already on the Hub with the same content are not re-sent."""\n    hub = make_hub(spec, log=log)\n    if hub is None:\n        raise ValueError(\'spec has no hub_repo\')\n    work = Path(spec[\'work_dir\'])\n    rels = hub_files(work, hub.upload_state)\n    for name in rels:  # one commit per large file keeps each commit retryable\n        if name.endswith((\'state.pt\', \'adapter_model.safetensors\')):\n            hub.upload(work, [name], f\'sync {name}\')\n    return hub.upload(work, [r for r in rels if not r.endswith((\'state.pt\', \'adapter_model.safetensors\'))], \'sync small files\')\n\n\nclass Run:\n    """One fine-tuning run (dataset, lr) with a fixed cfg[\'epochs\'] cosine schedule; resumable at optimizer steps.\n    After every epoch the model is exported to <out_dir>/epoch<k>/ and, with a hub, uploaded with the resume state."""\n\n    def __init__(self, c, cfg, dataset, lr, splits, pixels, out_dir, log=print, hub=None):\n        import torch\n        from transformers import get_cosine_schedule_with_warmup\n        self.c, self.cfg, self.dataset, self.lr, self.splits, self.pixels, self.log = c, cfg, dataset, lr, splits, pixels, log\n        self.out, self.hub = Path(out_dir), hub\n        self.work = self.out.parent.parent  # <work>/runs/<name>\n        self.examples = []\n        for r in splits[\'train\']:\n            ids, labels = build_train_example(c.tokenizer, r[\'question\'], r[\'answer\'], c.n_patches, cfg[\'max_length\'])\n            self.examples.append({\'input_ids\': ids, \'labels\': labels, \'image_name\': r[\'image_name\']})\n        self.steps_per_epoch = math.ceil(len(self.examples) / cfg[\'global_batch\'])\n        self.total_steps = self.steps_per_epoch * cfg[\'epochs\']\n        reset_trainable(c, cfg[\'seed\'])\n        self.params = trainable_named(c)\n        lora = [p for n, p in self.params if n.startswith(\'lora.\')]\n        proj = [p for n, p in self.params if n.startswith(\'projector.\')]\n        self.optimizer = torch.optim.AdamW([{\'params\': lora, \'lr\': lr, \'weight_decay\': cfg[\'weight_decay\']},\n                                            {\'params\': proj, \'lr\': cfg[\'projector_lr\'], \'weight_decay\': cfg[\'weight_decay\']}],\n                                           betas=(0.9, 0.999), eps=1e-8)\n        self.scheduler = get_cosine_schedule_with_warmup(self.optimizer, math.ceil(self.total_steps * cfg[\'warmup_ratio\']), self.total_steps)\n        self.state = {\'epoch\': 0, \'step_in_epoch\': 0, \'global_step\': 0, \'history\': [], \'losses\': []}\n        self._resume()\n\n    def _resume(self):\n        import numpy as np\n        import torch\n        path = self.out / \'state.pt\'\n        if not path.exists():\n            return\n        saved = torch.load(path, map_location=\'cpu\', weights_only=False)\n        named = dict(self.params)\n        for n, t in saved[\'params\'].items():\n            named[n].data.copy_(t.to(named[n].device, named[n].dtype))\n        self.optimizer.load_state_dict(saved[\'optimizer\'])\n        self.scheduler.load_state_dict(saved[\'scheduler\'])\n        self.state = saved[\'state\']\n        torch.set_rng_state(saved[\'rng\']); random.setstate(saved[\'py_rng\']); np.random.set_state(saved[\'np_rng\'])\n        if saved.get(\'cuda_rng\') is not None and torch.cuda.is_available():\n            torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        self.log(f\'resumed {self.out.name}: epoch {self.state["epoch"]}, step {self.state["step_in_epoch"]}/{self.steps_per_epoch}\')\n\n    def save(self):\n        import numpy as np\n        import torch\n        self.out.mkdir(parents=True, exist_ok=True)\n        saved = {\'params\': {n: p.detach().cpu() for n, p in self.params}, \'optimizer\': self.optimizer.state_dict(),\n                 \'scheduler\': self.scheduler.state_dict(), \'state\': self.state, \'rng\': torch.get_rng_state(),\n                 \'py_rng\': random.getstate(), \'np_rng\': np.random.get_state(),\n                 \'cuda_rng\': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}\n        torch.save(saved, self.out / \'state.tmp\')\n        (self.out / \'state.tmp\').replace(self.out / \'state.pt\')\n        _write_json(self.out / \'progress.json\', {**self.state, \'steps_per_epoch\': self.steps_per_epoch, \'total_steps\': self.total_steps,\n                                                 \'lr\': self.lr, \'dataset\': self.dataset, \'updated\': time.time()})\n\n    def train_until(self, epoch_end, on_epoch_end=None):\n        """Train until cfg epoch index `epoch_end` (1-based count of finished epochs) is reached."""\n        import torch\n        cfg, c = self.cfg, self.c\n        micro = min(cfg[\'micro_batch\'], cfg[\'global_batch\'])\n        while self.state[\'epoch\'] < epoch_end:\n            epoch = self.state[\'epoch\']\n            order = torch.randperm(len(self.examples), generator=torch.Generator().manual_seed(cfg[\'seed\'] * 1000 + epoch)).tolist()\n            c.model.train()\n            t0, steps_timed = time.time(), 0\n            for step in range(self.state[\'step_in_epoch\'], self.steps_per_epoch):\n                batch = [self.examples[i] for i in order[step * cfg[\'global_batch\']:(step + 1) * cfg[\'global_batch\']]]\n                chunks = [batch[i:i + micro] for i in range(0, len(batch), micro)]\n                self.optimizer.zero_grad(set_to_none=True)\n                total = 0.0\n                for chunk in chunks:  # HF Trainer: mean token loss per micro-batch / accumulation steps\n                    ids, labels, mask, pix = _collate(c, chunk, self.pixels)\n                    with _autocast(c):\n                        loss = c.model(inputs_embeds=embed(c, ids, pix), attention_mask=mask, labels=labels).loss / len(chunks)\n                    loss.backward()\n                    total += float(loss)\n                torch.nn.utils.clip_grad_norm_([p for _, p in self.params], cfg[\'max_grad_norm\'])\n                self.optimizer.step(); self.scheduler.step()\n                self.state[\'step_in_epoch\'] = step + 1\n                self.state[\'global_step\'] += 1\n                self.state[\'losses\'].append(round(total, 5))\n                steps_timed += 1\n                sec = (time.time() - t0) / steps_timed\n                remaining = (self.steps_per_epoch - step - 1) + (epoch_end - epoch - 1) * self.steps_per_epoch\n                self.log(f\'{self.dataset} lr={self.lr:g} epoch {epoch + 1} step {step + 1}/{self.steps_per_epoch} \'\n                         f\'loss {total:.4f} lr_now {self.scheduler.get_last_lr()[0]:.2e} {sec:.1f}s/step \'\n                         f\'eta(train to epoch {epoch_end}) {remaining * sec / 60:.1f} min\')\n                if not math.isfinite(total):\n                    raise FloatingPointError(f\'non-finite loss at {self.dataset} lr={self.lr} step {self.state["global_step"]}\')\n                if self.state[\'step_in_epoch\'] % cfg[\'save_every_steps\'] == 0 and self.state[\'step_in_epoch\'] < self.steps_per_epoch:\n                    self.save()\n            self.state[\'epoch\'] += 1\n            self.state[\'step_in_epoch\'] = 0\n            record = {\'epoch\': self.state[\'epoch\'], \'mean_loss\': sum(self.state[\'losses\'][-self.steps_per_epoch:]) / self.steps_per_epoch}\n            model_dir = self.out / f\'epoch{self.state["epoch"]}\'\n            export_model(c, model_dir, {\'dataset\': self.dataset, \'lr\': self.lr, \'epoch\': self.state[\'epoch\'],\n                                        \'schedule_epochs\': cfg[\'epochs\'], \'global_step\': self.state[\'global_step\'],\n                                        \'mean_loss\': record[\'mean_loss\'], \'config\': cfg})\n            self.log(f\'saved model {model_dir}\')\n            if on_epoch_end:\n                record.update(on_epoch_end(self, self.state[\'epoch\']) or {})\n            self.state[\'history\'].append(record)\n            self.save()\n            self.log(json.dumps({\'run\': self.out.name, **record}))\n            if self.hub:\n                rel = self.out.relative_to(self.work).as_posix()\n                self.hub.upload(self.work, [f\'{rel}/epoch{self.state["epoch"]}/{f}\' for f in MODEL_FILES],\n                                f\'{self.out.name} epoch {self.state["epoch"]} model\')\n                self.hub.upload(self.work, [f\'{rel}/state.pt\', f\'{rel}/progress.json\'], f\'{self.out.name} epoch {self.state["epoch"]} state\')\n\n\n# ---- pipeline --------------------------------------------------------------------------------------------------\ndef config_for(overrides=None):\n    return {**DEFAULTS, **(overrides or {})}\n\n\ndef lr_tag(lr):\n    return f\'{lr:.0e}\'.replace(\'e-0\', \'e-\')\n\n\ndef _free(run):\n    """Drop a finished run\'s optimizer state (~4 GB for r=128) before the next run is created."""\n    import gc\n    import torch\n    run.optimizer = run.scheduler = None\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef evaluate_split(c, cfg, splits, pixels, split, mode, results_dir, name, seed=0, log=print, hub=None):\n    path = Path(results_dir) / f\'{name}.json\'\n    if path.exists():\n        return json.loads(path.read_text(encoding=\'utf-8\'))[\'metrics\']\n    rows = splits[split]\n    preds, raw = generate(c, rows, pixels, mode, cfg, seed=seed, log=log)\n    metrics, per_question = score(rows, preds)\n    for item in per_question:\n        item[\'raw_output\'] = raw[str(item[\'qid\'])]\n    metrics.update(split=split, decoding=mode, questions=len(rows))\n    _write_json(path, {\'metrics\': metrics, \'per_question\': per_question})\n    log(f\'RESULT {name} \' + json.dumps({\'paper\': metrics[\'paper\'], \'unified\': metrics[\'unified\']}))\n    if hub:\n        hub.upload(path.parent.parent, [f\'results/{name}.json\'], f\'result {name}\')\n    return metrics\n\n\ndef run_pipeline(spec, components=None, log=print):\n    """Stages (each skipped when its result exists): zero-shot test, RAD lr sweep (1 epoch, validation), continue\n    the chosen lr to epoch 3 + test, SLAKE 3 epochs with the chosen lr + test."""\n    cfg = config_for(spec.get(\'overrides\'))\n    work, cache = Path(spec[\'work_dir\']), Path(spec[\'cache_dir\'])\n    res = work / \'results\'\n    hub = make_hub(spec, log=log)\n    if hub:  # before the 52 GB download: a bad repo/token fails fast, and a new runtime resumes from the Hub copy\n        restored = hub.restore(work)\n        log(f\'hub {hub.repo}/{hub.prefix}: restored {len(restored)} files\')\n    c = components or build_components(cfg, cache)\n    data = {d: load_data(d, work, cache, cfg, rad_dir=spec.get(\'rad_dir\')) for d in (\'rad\', \'slake\') if d in spec[\'datasets\']}\n    if spec.get(\'limit\'):  # smoke tests only\n        n = spec[\'limit\']\n        data = {d: {k: v[:n[k]] for k, v in s.items()} for d, s in data.items()}\n    for d, s in data.items():\n        log(f\'data {d}: \' + json.dumps(data_summary(s)))\n    pixels = {d: pixel_cache(c, [r for v in s.values() for r in v]) for d, s in data.items()}\n    test_modes = [\'paper\'] + ([\'greedy\'] if spec.get(\'test_greedy\', True) else [])\n\n    for d in spec.get(\'zero_shot\', []):\n        if d in data:\n            evaluate_split(c, cfg, data[d], pixels[d], \'test\', \'paper\', res, f\'zeroshot_{d}_test_paper\', log=log, hub=hub)\n\n    if \'rad\' in data:\n        rad = data[\'rad\']\n        val_hook = lambda run, epoch: {\'validation\': evaluate_split(\n            c, cfg, rad, pixels[\'rad\'], \'validation\', \'greedy\', res, f\'rad_lr{lr_tag(run.lr)}_epoch{epoch}_val_greedy\', log=log, hub=hub)[\'unified\']}\n        sweep = {}\n        for lr in cfg[\'lrs\']:\n            run = Run(c, cfg, \'rad\', lr, rad, pixels[\'rad\'], work / \'runs\' / f\'rad_lr{lr_tag(lr)}\', log=log, hub=hub)\n            run.train_until(cfg[\'select_epochs\'], on_epoch_end=val_hook)\n            sweep[lr] = evaluate_split(c, cfg, rad, pixels[\'rad\'], \'validation\', \'greedy\', res,\n                                       f\'rad_lr{lr_tag(lr)}_epoch{cfg["select_epochs"]}_val_greedy\', log=log, hub=hub)\n            _free(run)\n        chosen = spec.get(\'chosen_lr\') or max(cfg[\'lrs\'], key=lambda lr: (selection_key(sweep[lr]), -lr))\n        _write_json(res / \'lr_selection.json\', {\'chosen_lr\': chosen, \'manual\': bool(spec.get(\'chosen_lr\')),\n                                                \'rule\': \'validation unified overall_em, then open_f1, then smaller lr\',\n                                                \'validation_after_epochs\': cfg[\'select_epochs\'],\n                                                \'sweep\': {lr_tag(lr): sweep[lr] for lr in cfg[\'lrs\']}})\n        log(f\'chosen lr {chosen:g}\')\n        if hub:\n            hub.upload(work, [\'results/lr_selection.json\'], \'lr selection\')\n        run = Run(c, cfg, \'rad\', chosen, rad, pixels[\'rad\'], work / \'runs\' / f\'rad_lr{lr_tag(chosen)}\', log=log, hub=hub)\n        run.train_until(cfg[\'epochs\'], on_epoch_end=val_hook)\n        for mode in test_modes:\n            evaluate_split(c, cfg, rad, pixels[\'rad\'], \'test\', mode, res, f\'rad_lr{lr_tag(chosen)}_epoch{cfg["epochs"]}_test_{mode}\', log=log, hub=hub)\n        _free(run)\n    else:\n        chosen = spec.get(\'chosen_lr\')\n\n    if \'slake\' in data:\n        if chosen is None:\n            sel = res / \'lr_selection.json\'\n            chosen = json.loads(sel.read_text(encoding=\'utf-8\'))[\'chosen_lr\'] if sel.exists() else None\n        if chosen is None:\n            raise ValueError(\'SLAKE needs chosen_lr (run VQA-RAD first or set CHOSEN_LR)\')\n        slake = data[\'slake\']\n        run = Run(c, cfg, \'slake\', chosen, slake, pixels[\'slake\'], work / \'runs\' / f\'slake_lr{lr_tag(chosen)}\', log=log, hub=hub)\n        run.train_until(cfg[\'epochs\'])\n        for mode in test_modes:\n            evaluate_split(c, cfg, slake, pixels[\'slake\'], \'test\', mode, res, f\'slake_lr{lr_tag(chosen)}_epoch{cfg["epochs"]}_test_{mode}\', log=log, hub=hub)\n        if spec.get(\'slake_validation\'):\n            evaluate_split(c, cfg, slake, pixels[\'slake\'], \'validation\', \'greedy\', res, f\'slake_lr{lr_tag(chosen)}_epoch{cfg["epochs"]}_val_greedy\', log=log, hub=hub)\n    summary = summarize(work)\n    _write_json(work / \'summary.json\', summary)\n    if hub:\n        hub.upload(work, [\'summary.json\', \'spec.json\', \'train.log\'], \'summary\')\n    return summary\n\n\ndef summarize(work_dir):\n    res = Path(work_dir) / \'results\'\n    rows = {}\n    for path in sorted(res.glob(\'*.json\')):\n        if path.name == \'lr_selection.json\':\n            continue\n        rows[path.stem] = json.loads(path.read_text(encoding=\'utf-8\'))[\'metrics\']\n    sel = res / \'lr_selection.json\'\n    return {\'results\': rows, \'lr_selection\': json.loads(sel.read_text(encoding=\'utf-8\')) if sel.exists() else None,\n            \'paper_open_closed_overall\': PAPER, \'deviations\': DEVIATIONS, \'config\': DEFAULTS}\n\n\n# ---- background runner (same pattern as biomedclip_meter) ------------------------------------------------------------\nDONE_MARK, START_MARK = \'ALL_STAGES_DONE\', \'=== pipeline start\'\n\n\ndef _pid_alive(pid):\n    try:\n        stat = Path(f\'/proc/{pid}/stat\').read_text()\n        cmdline = Path(f\'/proc/{pid}/cmdline\').read_bytes()\n    except OSError:\n        return False\n    return stat.rsplit(\')\', 1)[1].split()[0] != \'Z\' and b\'biomed_vital_qlora\' in cmdline\n\n\ndef trainer_pid(work_dir):\n    path = Path(work_dir) / \'train.pid\'\n    if path.exists():\n        pid = int(path.read_text().strip())\n        if _pid_alive(pid):\n            return pid\n    return None\n\n\ndef start_trainer(spec):\n    work = Path(spec[\'work_dir\'])\n    work.mkdir(parents=True, exist_ok=True)\n    pid = trainer_pid(work)\n    if pid:\n        print(\'pipeline already running, pid\', pid)\n        return pid\n    _write_json(work / \'spec.json\', spec)\n    with open(work / \'train.log\', \'a\', encoding=\'utf-8\') as log:\n        log.write(f\'\\n{START_MARK} {time.strftime("%Y-%m-%d %H:%M:%S")} ===\\n\')\n        log.flush()\n        proc = subprocess.Popen([sys.executable, \'-u\', str(Path(__file__).resolve()), str(work / \'spec.json\')],\n                                cwd=str(Path(__file__).resolve().parent), stdout=log, stderr=subprocess.STDOUT,\n                                stdin=subprocess.DEVNULL, start_new_session=True, env=dict(os.environ, PYTHONUNBUFFERED=\'1\'))\n    (work / \'train.pid\').write_text(str(proc.pid))\n    print(\'pipeline started, pid\', proc.pid, \'| log\', work / \'train.log\')\n    return proc.pid\n\n\ndef stop_trainer(work_dir):\n    import signal\n    pid = trainer_pid(work_dir)\n    if pid:\n        os.killpg(pid, signal.SIGTERM)\n        print(\'sent SIGTERM to\', pid, \'(restart resumes from the last saved step)\')\n    else:\n        print(\'no pipeline running\')\n\n\ndef trainer_status(work_dir, tail_lines=15):\n    log = Path(work_dir) / \'train.log\'\n    text = log.read_text(encoding=\'utf-8\', errors=\'replace\') if log.exists() else \'\'\n    latest = text.rsplit(START_MARK, 1)[-1] if START_MARK in text else \'\'\n    state = \'running\' if trainer_pid(work_dir) else \'finished\' if DONE_MARK in latest else \'failed\' if latest else \'not started\'\n    return {\'state\': state, \'tail\': \'\\n\'.join(latest.strip().splitlines()[-tail_lines:])}\n\n\ndef progress(work_dir):\n    rows = []\n    for p in sorted(Path(work_dir).glob(\'runs/*/progress.json\')):\n        s = json.loads(p.read_text(encoding=\'utf-8\'))\n        rows.append({\'run\': p.parent.name, \'epoch_done\': s[\'epoch\'], \'step_in_epoch\': f"{s[\'step_in_epoch\']}/{s[\'steps_per_epoch\']}",\n                     \'global_step\': f"{s[\'global_step\']}/{s[\'total_steps\']}", \'last_loss\': s[\'losses\'][-1] if s[\'losses\'] else None,\n                     \'min_since_save\': round((time.time() - s[\'updated\']) / 60, 1)})\n    return rows\n\n\nBACKUP_BEGIN, BACKUP_END = \'===QCR_BACKUP_BEGIN===\', \'===QCR_BACKUP_END===\'\n\n\ndef backup_text(work_dir, max_log_bytes=1 << 20):\n    """results/*.json (metrics + per-question predictions), summary, spec, run histories and log tail, base64 zip."""\n    work = Path(work_dir)\n    buf = io.BytesIO()\n    with zipfile.ZipFile(buf, \'w\', compression=zipfile.ZIP_DEFLATED) as z:\n        for name in (\'summary.json\', \'spec.json\'):\n            if (work / name).exists():\n                z.write(work / name, name)\n        for p in sorted(work.glob(\'results/*.json\')) + sorted(work.glob(\'runs/*/progress.json\')):\n            z.write(p, p.relative_to(work).as_posix())\n        if (work / \'train.log\').exists():\n            z.writestr(\'train.log\', (work / \'train.log\').read_bytes()[-max_log_bytes:])\n    data = buf.getvalue()\n    b64 = base64.b64encode(data).decode(\'ascii\')\n    return \'\\n\'.join([BACKUP_BEGIN, f\'biomed_vital_backup.zip sha256={hashlib.sha256(data).hexdigest()}\',\n                      *[b64[i:i + 100] for i in range(0, len(b64), 100)], BACKUP_END])\n\n\nDEVIATIONS = [\n    \'QLoRA (4-bit NF4 LLaMA-13B + LoRA r128/alpha256 on all LLM linear layers, projector trained) instead of the \'\n    \'full-model fine-tuning of LLaVA-Med / BioMed-VITAL (8 GPUs); LoRA recipe from LLaVA v1.5 finetune_task_lora.sh\',\n    \'3 epochs (LLaVA-Med v1.0.0 fine-tuning command) instead of the paper\\\'s 15, because of compute\',\n    \'LoRA lr chosen from {2e-5, 1e-4, 2e-4} on a VQA-RAD validation split after 1 epoch of the 3-epoch schedule; \'\n    \'the paper\\\'s 2e-5 is a full fine-tuning lr. The projector keeps 2e-5. SLAKE reuses the VQA-RAD choice\',\n    \'VQA-RAD validation = 10% of the training images (seed 0), so VQA-RAD is trained on ~90% of the 1,797 training \'\n    \'questions; the paper has no validation split\',\n    \'Fine-tuning text format, image tokens and label masking follow LLaVA-Med v1.0.0 train.py (v0); the authors\\\' \'\n    \'downstream fine-tuning data/script are not public. Human turn = "<question>\\\\n<image>"\',\n    \'Generated text: a leading "###" is dropped before cutting at the first "###" (training ends the human turn with \'\n    \'"\\\\n### Assistant:", the eval prompt with "###", so the fine-tuned model can open with "###"); raw outputs are saved\',\n    \'Validation uses greedy decoding (max 128 new tokens) for a deterministic choice; the test set is decoded with \'\n    \'the authors\\\' settings (sampling, T=0.7, 1024 new tokens, seed 0) and additionally greedily\',\n    \'Paper metrics = LLaVA-Med run_eval.py (closed: yes/no hit, open: token recall). Unified metrics = normalised \'\n    \'exact match (closed accuracy, open EM) and token F1 after removing a leading "Assistant:" tag, Open/Closed by \'\n    \'the ground-truth answer_type, for comparison with the classification baselines\',\n    \'Generation is batched with left padding (bf16 autocast); the authors generate one question at a time in fp16\',\n]\n\n\nif __name__ == \'__main__\':  # background entry point used by start_trainer\n    spec = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\n    run_pipeline(spec)\n    print(DONE_MARK, time.strftime(\'%Y-%m-%d %H:%M:%S\'), flush=True)\n', encoding='utf-8')
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
import qcr_port, biomedclip_meter, biomed_vital_qlora as bv
importlib.reload(qcr_port); importlib.reload(biomedclip_meter); importlib.reload(bv)
print(bv.MODEL, bv.VISION)


## 3. 설정

- `DATASETS`: `['rad', 'slake']`. SLAKE만 따로 돌리려면 `['slake']`와 `CHOSEN_LR`을 함께 지정하세요.
- `LRS`: 비교할 LoRA lr (교수님 지시: 2e-5, 1e-4, 2e-4)
- `CHOSEN_LR`: `None`이면 validation으로 자동 선택합니다(통일 overall EM → open F1 → 작은 lr 순). 숫자를 넣으면 그 lr을 씁니다.
- `ZERO_SHOT`: fine-tuning 전 test 평가를 할 데이터셋(기본 끔). 학습 전 모델은 답을 길게(최대 1024 토큰) 생성하므로
  VQA-RAD만으로 20~60분 걸릴 수 있습니다. 기준점이 필요하면 `['rad']`.
- `TEST_GREEDY`: test를 저자 방식(샘플링) 외에 greedy로도 한 번 더 평가합니다(재현성 확인용).
- `MICRO_BATCH`: GPU에 한 번에 올리는 개수. 전역 batch 64는 gradient accumulation으로 유지됩니다. 메모리가 부족하면 2로 줄이세요.
- `QUICK_CHECK`: `True`면 데이터 일부(train 128개)로 전체 흐름만 확인합니다(약 20~30분, 결과는 의미 없음). 별도 폴더에 저장됩니다.
- `HUB_REPO`: 체크포인트·모델을 올릴 **비공개** HF model repo(`본인아이디/이름`). 없으면 다음 셀이 비공개로 만듭니다. `None`이면 Colab 디스크에만 저장됩니다.
- `HUB_STATE`: `True`면 `state.pt`(약 6GB)도 epoch마다 올립니다(새 런타임에서 이어서 학습하려면 필요).
  같은 경로에 덮어써도 HF는 이전 버전을 기록에 남기므로 전체 실행에서 repo가 수십 GB까지 커질 수 있습니다. 모델만 필요하면 `False`.


In [ ]:
import json
DATASETS = ['rad', 'slake']
LRS = [2e-5, 1e-4, 2e-4]
CHOSEN_LR = None
ZERO_SHOT = []
TEST_GREEDY = True
MICRO_BATCH = 4
QUICK_CHECK = False
HUB_REPO = None  # 예: 'my-hf-id/biomed-vital-qlora'
HUB_STATE = True
CACHE = Path('/content/hf_cache')
WORK = Path('/content/biomed_vital_quick' if QUICK_CHECK else '/content/biomed_vital')
WORK.mkdir(parents=True, exist_ok=True)
assert set(DATASETS) <= {'rad', 'slake'} and DATASETS
SPEC = {'work_dir': str(WORK), 'cache_dir': str(CACHE), 'datasets': DATASETS, 'zero_shot': ZERO_SHOT,
        'chosen_lr': CHOSEN_LR, 'test_greedy': TEST_GREEDY, 'overrides': {'lrs': LRS, 'micro_batch': MICRO_BATCH}}
if QUICK_CHECK:
    SPEC['limit'] = {'train': 128, 'validation': 24, 'test': 24}
if HUB_REPO:  # Hub 안에서도 WORK 폴더 이름(biomed_vital / biomed_vital_quick) 아래에 저장
    SPEC.update(hub_repo=HUB_REPO, hub_prefix=WORK.name, hub_state=HUB_STATE)
else:
    print('경고: HUB_REPO가 없어서 모델·체크포인트는 Colab 디스크에만 남고 런타임이 끝나면 사라집니다.')
print(json.dumps(SPEC, indent=1))


### 3-1. Hugging Face 로그인 (HUB_REPO를 쓸 때)

**write** 권한 토큰이 필요합니다(https://huggingface.co/settings/tokens). Colab 보안 비밀(`HF_TOKEN`)이 있으면 그것을 쓰고,
없으면 입력창이 뜹니다. 토큰은 환경변수로만 학습 프로세스에 전달되고 `spec.json`이나 출력에는 남지 않습니다.


In [ ]:
import os, getpass
from huggingface_hub import HfApi
if HUB_REPO:
    if not os.environ.get('HF_TOKEN'):
        try:
            from google.colab import userdata
            os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        except Exception:
            os.environ['HF_TOKEN'] = getpass.getpass('HF write token: ').strip()
    api = HfApi()
    print('logged in as', api.whoami()['name'])
    url = api.create_repo(HUB_REPO, repo_type='model', private=True, exist_ok=True)
    info = api.model_info(HUB_REPO)
    assert info.private, f'{HUB_REPO} is PUBLIC - use a private repo'
    print('hub repo (private):', url)
else:
    print('HUB_REPO = None: Hub 저장 안 함')


## 4. 데이터 확인

VQA-RAD 원본 split과 validation 분할, SLAKE 영어 split의 개수를 확인합니다.
토크나이저만 받아서 학습 예시 하나의 **학습 대상 토큰**과 평가 프롬프트를 보여줍니다.


In [ ]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer
for d in DATASETS:
    print(d, json.dumps(bv.data_summary(bv.load_data(d, WORK, CACHE, bv.config_for(SPEC['overrides']))), indent=1))
tok_dir = Path(hf_hub_download(bv.MODEL[0], 'tokenizer.model', revision=bv.MODEL[1], cache_dir=str(CACHE))).parent
for f in ('tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json'):
    hf_hub_download(bv.MODEL[0], f, revision=bv.MODEL[1], cache_dir=str(CACHE))
tok = AutoTokenizer.from_pretrained(tok_dir, use_fast=False)
ids, labels = bv.build_train_example(tok, 'Is there a rib fracture?', 'No', 576, 2048)
print('train tokens', len(ids), '| supervised:', repr(tok.decode([t for t, l in zip(ids, labels) if l != bv.IGNORE_INDEX])))
print('eval prompt (image tokens shortened):')
print(bv.build_eval_prompt('Is there a rib fracture?', 3))


## 5. 학습 + 평가 (백그라운드)

### 5-1. 시작

처음에는 약 52GB 모델을 받고 4bit로 올리는 데 20~30분 걸립니다(로그에 진행 표시).
이미 실행 중이면 새로 띄우지 않고, 끝난 단계는 건너뜁니다. 멈추려면 `bv.stop_trainer(WORK)`.
`HUB_REPO`가 있으면 모델을 받기 전에 Hub에 있는 체크포인트·결과를 먼저 내려받고(`hub: restored ...`),
epoch이 끝날 때마다 올린 기록이 로그에 `hub: ...`로 찍힙니다(epoch당 1~3분). 업로드가 실패해도 학습은 계속되고, 8번 셀로 다시 올릴 수 있습니다.


In [ ]:
bv.start_trainer(SPEC)


### 5-2. 진행 상황 (언제든 실행 가능)

학습 로그의 `s/step`과 `eta`로 남은 시간을 볼 수 있습니다. VQA-RAD는 epoch당 26 step, SLAKE는 77 step입니다.


In [ ]:
import subprocess
import pandas as pd
from IPython.display import display
status = bv.trainer_status(WORK, tail_lines=20)
print('pipeline:', status['state'])
rows = bv.progress(WORK)
if rows: display(pd.DataFrame(rows))
print(subprocess.run('nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout)
print(status['tail'])


### 5-3. 끝날 때까지 대기

10분마다 로그 마지막 줄을 출력합니다. 이 셀을 중단해도 학습은 계속됩니다.


In [ ]:
import time
while bv.trainer_pid(WORK):
    tail = bv.trainer_status(WORK, tail_lines=1)['tail']
    print(time.strftime('%H:%M:%S'), tail[-200:], flush=True)
    time.sleep(600)
status = bv.trainer_status(WORK, tail_lines=40)
if status['state'] != 'finished':
    raise RuntimeError('pipeline ended without finishing:\n' + status['tail'])
print('all stages finished')


## 6. 결과

- **논문 방식**: LLaVA-Med `run_eval.py` (closed: 답에 yes/no가 있고 정답이 포함되면 정답, open: 정답 토큰 recall). 논문 수치와 비교하는 값입니다.
- **통일 방식**: 앞의 `Assistant:` 태그만 떼고 정규화한 exact match(closed 정확도, open EM)와 token F1. 분류 모델(PubMedCLIP, BiomedCLIP)과 비교하는 값입니다.


In [ ]:
from IPython.display import Markdown, display
summary = bv.summarize(WORK)
(WORK / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
if summary['lr_selection']:
    print('chosen lr:', summary['lr_selection']['chosen_lr'], '| rule:', summary['lr_selection']['rule'])
lines = ['|Result|Split|Decoding|Open (paper)|Closed (paper)|Overall (paper)|Closed acc|Open EM|Open F1|Overall EM|',
         '|---|---|---|---:|---:|---:|---:|---:|---:|---:|']
for name, m in summary['results'].items():
    p, u = m['paper'], m['unified']
    lines.append(f"|{name}|{m['split']}|{m['decoding']}|{p['open']:.2f}|{p['closed']:.2f}|{p['overall']:.2f}|"
                 f"{u['closed_acc']:.2f}|{u['open_em']:.2f}|{u['open_f1']:.2f}|{u['overall_em']:.2f}|")
for d, (o, c, a) in bv.PAPER.items():
    lines.append(f'|**paper** {d}|test|sampling|{o:.2f}|{c:.2f}|{a:.2f}|||||')
display(Markdown('\n'.join(lines)))
print('근사 실험: QLoRA, 3 epoch, lr은 validation 선택. 차이는 summary.json의 deviations 참고.')


## 7. 결과 백업 (노트북 출력에 저장)

결과 JSON(질문별 예측 포함), 진행 기록, 로그 끝부분을 zip으로 묶어 base64로 출력합니다. LoRA 가중치(수 GB)는 들어가지 않습니다.
**셀이 끝나면 노트북을 저장(Ctrl+S)**한 뒤 로컬에서 다음을 실행하면 파일이 복원됩니다.

```
python extract_notebook_backup.py notebooks/BioMedVITAL_QLoRA_Approximation.ipynb colab_backup/biomed_vital
```


In [ ]:
backup = bv.backup_text(WORK)
print(f'{len(backup) / 1e6:.2f} MB of text')
print(backup)


## 8. 체크포인트·모델을 PC로 받기 (Hugging Face Hub)

### 8-1. Hub에 빠진 파일 다시 올리기 (Colab에서)

학습 중 업로드가 실패했거나 마지막 로그까지 올리고 싶을 때 실행합니다. 이미 같은 내용이 있는 파일은 다시 보내지 않습니다.


In [ ]:
assert HUB_REPO, '3번 셀에서 HUB_REPO를 지정하세요'
local = bv.hub_files(WORK, HUB_STATE)
print(len(local), 'files on the Colab disk')
assert bv.hub_sync(SPEC), 'upload failed - 로그를 보고 다시 실행하세요'
remote = [f for f in HfApi().list_repo_files(HUB_REPO) if f.startswith(WORK.name + '/')]
missing = sorted(set(local) - {f[len(WORK.name) + 1:] for f in remote})
print(len(remote), 'files on the Hub under', WORK.name + '/', '| missing:', missing or 'none')
for f in sorted(remote):
    if f.endswith(('adapter_model.safetensors', 'state.pt')):
        print(' ', f)


### 8-2. PC에서 받기 (VS Code 터미널, Colab 커널이 아니라 로컬에서 실행)

```
pip install -U huggingface_hub
hf auth login
hf download <HUB_REPO> --local-dir colab_backup/biomed_vital_hub --exclude "*state.pt"
```

- 모델만 받습니다(run·epoch마다 약 2GB). 이어서 학습할 체크포인트(`state.pt`, 약 6GB)까지 받으려면 `--exclude` 부분을 지우세요.
- 특정 run만: `--include "biomed_vital/runs/rad_lr2e-4/epoch3/*" "biomed_vital/results/*"`
- 받은 모델은 Colab에서 모델을 올린 뒤(`c = bv.build_components(...)`) `bv.load_model(c, '<폴더>/epoch3')`로 다시 넣어 평가할 수 있습니다.
